# Reasoning Trajectories, Phase Transitions and Answer Development

**Goal.** Understand how LLM reasoning traces evolve: what extra reasoning contributes, how discourse phases change along a trace, how answers emerge and change, and how successful and unsuccessful attempts differ. 

**Principles used throughout**
* Segments are reconstructed from `thinking_text` markers and matched to `rtl_labels_json` by `segment_id`; **original order is never changed**.
* Only the *existing* annotations (`discourse_role`, `answer_status`, `answer_value`, `temporal_role`) are used. No new labels, no LLM calls.
* **Word counts** (whitespace-delimited tokens after replacing `<br>` tags with spaces and removing zero-width characters) are the primary length measure. Token counts are only supplementary.
* Comparisons are made **within question** (key = `benchmark::question_id`) and, where possible, **within compute level**. Statistics are computed at the question level (segments and samples of one question are not independent). All transition/pattern results are exploratory and **none are causal**.
* Two kinds of trajectory figure are kept strictly apart: **absolute** maps (x = cumulative words, shared scale, widths proportional to words) and **normalized** maps (x = 0–100 % of each trace).

## 1. Configuration and imports

In [ ]:
# ============ CONFIGURATION (edit here only) ============
CSV_PATH = "/kaggle/input/reasoning-traces/traces.csv"   # input CSV
OUT_DIR  = "/kaggle/working/outputs"                      # figures + tables are saved here

MIN_SAMPLES    = 4     # min eligible samples per question for group selection (auto-relaxed if a group would be too small)
LEN_RANGE_SD_K = 1.5   # Group 1: (max_len - min_len) >= K * within-question SD of word counts
LEN_MIN_RATIO  = 1.5   # Group 1: additionally max_len / min_len >= this ("highly different")
N_EXAMPLES     = 3     # representative questions drawn per group in trajectory figures
DIV_MIN_WORDS  = 40    # divergence detector: a structural difference must involve >= this many words
SEED           = 0
# ========================================================

import os, re, json, itertools, warnings
from difflib import SequenceMatcher
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import seaborn as sns
from scipy import stats
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
os.makedirs(OUT_DIR, exist_ok=True)
rng = np.random.default_rng(SEED)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 50); pd.set_option("display.max_colwidth", 110)

ROLES = ["setup", "reasoning", "doubt", "verification", "correction"]
KEY = ["doubt", "verification", "correction"]
ROLE_COLORS = {"setup": "#9aa5b1", "reasoning": "#4c78a8", "doubt": "#f2a541", "verification": "#54a24b", "correction": "#d6404e"}
COND_COLORS = {"G1 correct": "#2a9d8f", "G2 correct": "#4c78a8", "G2 incorrect": "#e45756", "G3 incorrect": "#7f3c8d"}
R = {}   # key numbers collected for the final summary

def save_fig(name): plt.savefig(f"{OUT_DIR}/{name}.png", dpi=130, bbox_inches="tight")
def save_tab(d, name, index=True): d.to_csv(f"{OUT_DIR}/{name}.csv", index=index)

## 2. Dataset loading and initial inspection

In [ ]:
REQUIRED = ["question_id", "benchmark", "difficulty_tier", "task_type", "compute_level", "sample_index", "raw_model_output",
            "thinking_text", "gold", "is_correct", "tokens_used", "thinking_tokens", "output_tokens", "hit_response_marker",
            "was_truncated", "hit_marker_empty_output", "rtl_status", "rtl_n_segments", "rtl_labels_json"]
df = pd.read_csv(CSV_PATH, low_memory=False)
missing_cols = [c for c in REQUIRED if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")
df = df.reset_index(drop=True)
df["trace_id"] = df.index            # internal row id; source columns are left untouched

_FLAG = {"true": 1.0, "1": 1.0, "1.0": 1.0, "yes": 1.0, "false": 0.0, "0": 0.0, "0.0": 0.0, "no": 0.0}
def to_flag(s):   # -> 1.0 / 0.0 / NaN (unrecognised or missing values stay NaN, never "False")
    return s.map(lambda v: _FLAG.get(str(v).strip().lower(), np.nan) if pd.notna(v) else np.nan).astype(float)

df["correct"]    = to_flag(df.is_correct)
df["trunc"]      = to_flag(df.was_truncated)
df["marker_hit"] = to_flag(df.hit_response_marker)
df["empty_out"]  = to_flag(df.hit_marker_empty_output)
df["compute"]    = df.compute_level.astype(str)
df["qkey"]       = df.benchmark.astype(str) + "::" + df.question_id.astype(str)
df["id_ok"]      = df.question_id.notna() & df.benchmark.notna()
df["dup_repeat"] = df.duplicated(["qkey", "compute", "sample_index"], keep="first")

span = df.groupby("question_id").benchmark.nunique()
print(f"{len(df):,} traces | {df.qkey.nunique():,} questions (benchmark::question_id)")
print(f"question_id values shared by >1 benchmark: {(span > 1).sum()} of {len(span)}  -> the composite key `qkey` is used for all grouping")
display(df.groupby(["benchmark", "compute"]).size().unstack(fill_value=0))
display(pd.DataFrame({"missing": df[REQUIRED].isna().sum(), "pct": (df[REQUIRED].isna().mean() * 100).round(2)}).T)

## 3. Segment reconstruction and annotation matching

* `thinking_text` is split on the markers `⟦S<n>⟧`; the text after a marker belongs to that segment until the next marker. **Order = order of appearance in the text** (`seg_pos`), not marker number.
* Annotations are matched to segments by `segment_id` (digits of the id are compared, so `⟦S3⟧`/`S3` match; non-standard id formats are counted and reported).
* The raw segment text is kept untouched in `text`; only the word counter replaces `<br>` tags by spaces and strips zero-width characters.
* A segment without a matching annotation keeps empty label fields (**nothing is invented**). Labels outside the expected sets are kept as `*_raw`, counted, and *not* remapped; the cleaned columns (`role`, `answer_status`, `temporal_role`) are NaN for them.

In [ ]:
MARK = re.compile(r"(⟦S(\d+)⟧)")
BR   = re.compile(r"<\s*/?\s*br\s*/?\s*>", re.I)
ZW   = re.compile("[\u200b-\u200d\ufeff]")
MOJI = re.compile("â€|Ã[\x80-\xbf]|Â\xa0")
STATUS, TEMP = {"none", "candidate", "final"}, {"new", "continuation"}

def clean(t): return ZW.sub("", BR.sub(" ", t))
def wc(t): return len(clean(t).split())          # word count: whitespace-delimited tokens of cleaned text

def split_segments(text):
    parts = MARK.split(text)                       # [pre, marker, num, body, marker, num, body, ...]
    return parts[0], [(parts[k], int(parts[k + 1]), parts[k + 2]) for k in range(1, len(parts), 3)]

def parse_ann(s):
    if pd.isna(s) or not str(s).strip(): return "missing", []
    try: obj = json.loads(s)
    except json.JSONDecodeError: return "malformed", []
    if not (isinstance(obj, list) and all(isinstance(o, dict) for o in obj)): return "bad_structure", []
    return "ok", obj

def seg_num(sid):
    m = re.search(r"\d+", str(sid)); return int(m.group()) if m else None

seg_rows, tr_rows = [], []
for tid, text, ann_s in zip(df.trace_id, df.thinking_text, df.rtl_labels_json):
    text = "" if pd.isna(text) else str(text)
    pre, parts = split_segments(text)
    status, ann = parse_ann(ann_s)
    amap, dup_ann, nonstd, no_id = {}, 0, 0, 0
    for a in ann:
        n = seg_num(a.get("segment_id"))
        if n is None: no_id += 1; continue
        if n in amap: dup_ann += 1; continue          # first annotation for an id wins; duplicates are counted
        amap[n] = a
        nonstd += a.get("segment_id") != f"⟦S{n}⟧"
    nums = [n for _, n, _ in parts]
    for pos, (marker, n, body) in enumerate(parts):
        a = amap.get(n)
        seg_rows.append((tid, pos, marker, n, body, wc(body), a is not None,
                         a.get("discourse_role") if a else None, a.get("answer_status") if a else None,
                         a.get("answer_value") if a else None, a.get("temporal_role") if a else None))
    tr_rows.append(dict(trace_id=tid, n_seg_text=len(parts), pre_words=wc(pre), ann_status=status, n_ann=len(ann),
                        n_ann_noid=no_id, n_unmatched_ann=len(set(amap) - set(nums)), n_dup_ann=dup_ann, n_nonstd_id=nonstd,
                        n_dup_seg_id=len(nums) - len(set(nums)), ids_nonmonotone=any(b <= a for a, b in zip(nums, nums[1:])),
                        moji=bool(MOJI.search(text)), has_br=bool(BR.search(text))))

seg = pd.DataFrame(seg_rows, columns=["trace_id", "seg_pos", "seg_id", "seg_num", "text", "words", "has_ann",
                                      "role_raw", "status_raw", "answer_value", "temporal_raw"])
seg["role"]          = seg.role_raw.where(seg.role_raw.isin(ROLES))
seg["answer_status"] = seg.status_raw.where(seg.status_raw.isin(STATUS))
seg["temporal_role"] = seg.temporal_raw.where(seg.temporal_raw.isin(TEMP))
seg["is_empty"]   = seg.words == 0
seg["no_ann"]     = ~seg.has_ann
seg["bad_role"]   = seg.has_ann & seg.role.isna()
seg["bad_status"] = seg.has_ann & seg.answer_status.isna()
seg["bad_temp"]   = seg.has_ann & seg.temporal_role.isna()

tq = seg.groupby("trace_id").agg(total_words=("words", "sum"), n_empty_seg=("is_empty", "sum"), n_missing_ann=("no_ann", "sum"),
                                 n_bad_role=("bad_role", "sum"), n_bad_status=("bad_status", "sum"), n_bad_temp=("bad_temp", "sum")).reset_index()
df_t = df.merge(pd.DataFrame(tr_rows), on="trace_id").merge(tq, on="trace_id", how="left")
for c in ["total_words", "n_empty_seg", "n_missing_ann", "n_bad_role", "n_bad_status", "n_bad_temp"]:
    df_t[c] = df_t[c].fillna(0).astype(int)

# trace is "labeled" when segments were reconstructed, the annotation JSON parsed, and EVERY text segment has a valid discourse_role
df_t["labeled_ok"] = (df_t.ann_status == "ok") & (df_t.n_seg_text > 0) & (df_t.n_missing_ann == 0) & (df_t.n_bad_role == 0)
df_t["eligible"] = (df_t.labeled_ok & df_t.id_ok & ~df_t.dup_repeat & (df_t.trunc == 0) & df_t.correct.notna() & (df_t.total_words > 0))

# positions on the absolute (cumulative word) scale and relative scale
seg["end"] = seg.groupby("trace_id").words.cumsum()
seg["start"] = seg.end - seg.words
seg = seg.merge(df_t[["trace_id", "qkey", "benchmark", "difficulty_tier", "task_type", "compute", "sample_index", "correct", "eligible", "total_words"]],
                on="trace_id", how="left")
seg["rel_start"] = seg.start / seg.total_words.where(seg.total_words > 0)
seg["rel_end"] = seg.end / seg.total_words.where(seg.total_words > 0)
print(f"{len(seg):,} segments reconstructed from {len(df_t):,} traces; eligible traces: {int(df_t.eligible.sum()):,}")

## 4. Data-quality assessment

In [ ]:
def issue(label, mask, segs=None):
    segs = int(df_t.loc[mask, "n_seg_text"].sum()) if segs is None else int(segs)
    return dict(issue=label, traces=int(mask.sum()), pct_traces=round(100 * mask.mean(), 2), segments_affected=segs)

d = df_t
rows = [
    issue("missing question_id or benchmark", ~d.id_ok),
    issue("repeated (qkey, compute, sample_index) key - later copies", d.dup_repeat),
    issue("thinking_text missing/empty", d.thinking_text.isna() | (d.thinking_text.astype(str).str.strip() == "")),
    issue("no segment markers found", d.n_seg_text == 0),
    issue("text before the first marker", d.pre_words > 0),
    issue("marker numbers not increasing in text order (text order kept)", d.ids_nonmonotone),
    issue("annotation JSON missing", d.ann_status == "missing"),
    issue("annotation JSON malformed", d.ann_status == "malformed"),
    issue("annotation JSON has unexpected structure", d.ann_status == "bad_structure"),
    issue("rtl_n_segments missing", d.rtl_n_segments.isna()),
    issue("rtl_n_segments != segments in text", d.rtl_n_segments.notna() & (d.rtl_n_segments != d.n_seg_text)),
    issue("parsed annotations != segments in text", (d.ann_status == "ok") & (d.n_ann != d.n_seg_text)),
    issue("text segments without annotation", d.n_missing_ann > 0, d.n_missing_ann.sum()),
    issue("annotation ids absent from text", d.n_unmatched_ann > 0, d.n_unmatched_ann.sum()),
    issue("annotation entries without a usable segment_id", d.n_ann_noid > 0, d.n_ann_noid.sum()),
    issue("duplicate annotation ids", d.n_dup_ann > 0, d.n_dup_ann.sum()),
    issue("duplicate segment ids in text", d.n_dup_seg_id > 0, d.n_dup_seg_id.sum()),
    issue("non-standard segment_id format in annotations", d.n_nonstd_id > 0, d.n_nonstd_id.sum()),
    issue("unexpected/missing discourse_role", d.n_bad_role > 0, d.n_bad_role.sum()),
    issue("unexpected/missing answer_status", d.n_bad_status > 0, d.n_bad_status.sum()),
    issue("unexpected/missing temporal_role", d.n_bad_temp > 0, d.n_bad_temp.sum()),
    issue("empty (0-word) segments", d.n_empty_seg > 0, d.n_empty_seg.sum()),
    issue("is_correct missing/invalid", d.correct.isna()),
    issue("was_truncated = True", d.trunc == 1),
    issue("was_truncated missing/invalid", d.trunc.isna()),
    issue("hit_response_marker False/missing", d.marker_hit != 1),
    issue("extracted output empty (hit_marker_empty_output)", d.empty_out == 1),
    issue("possible encoding artifacts (mojibake)", d.moji),
    issue("contains <br> tags", d.has_br),
]
dq = pd.DataFrame(rows)
display(dq)
save_tab(dq, "data_quality", index=False)

print("rtl_status:", df.rtl_status.value_counts(dropna=False).to_dict())
for col, ok in [("role_raw", ROLES), ("status_raw", STATUS), ("temporal_raw", TEMP)]:
    bad = seg.loc[seg.has_ann & ~seg[col].isin(ok), col].astype(str).value_counts()
    print(f"unexpected {col} values:", bad.to_dict() if len(bad) else "none")

m = pd.Series(True, index=d.index); wf = [("all traces", int(m.sum()))]
for name, cond in [("identifiers present", d.id_ok), ("not a repeated key", ~d.dup_repeat),
                   ("segments reconstructed, JSON valid, every segment has a valid discourse_role", d.labeled_ok),
                   ("not truncated (flag known)", d.trunc == 0), ("correctness known", d.correct.notna()), ("non-empty text", d.total_words > 0)]:
    m = m & cond; wf.append((name, int(m.sum())))
assert m.sum() == d.eligible.sum()
display(pd.DataFrame(wf, columns=["step (cumulative)", "traces remaining"]))
R["n_total"], R["n_eligible"] = len(d), int(d.eligible.sum())

**How to read the quality report.** 
* Traces failing any *labeling* check (missing/malformed JSON, unlabeled segments, unexpected roles) are excluded from every trajectory analysis (they could only be analysed with invented labels). If these are non-randomly distributed (e.g. concentrated in long traces, one benchmark or incorrect samples) the eligible set is a **selected subset**; the share by `compute`/correctness is shown below.
* **Truncated** traces end mid-reasoning: their lengths, final phases and answer positions are censored. They are excluded from group analyses and examined separately (section 6).
* `hit_response_marker = False` or empty extracted outputs mean an answer may not have been extracted even if the thinking is complete; they are kept (they are real failures) but are reported per group later.
* Non-standard ids, duplicates, `rtl_n_segments` mismatches are *reported*, never silently repaired: segment matching used only the id found in the text.

In [ ]:
sel = d.assign(excluded_labeling=~d.labeled_ok)
display(pd.concat([sel.groupby("compute").excluded_labeling.mean().mul(100).round(1).rename("% not labeled-valid (by compute)"),
                   sel.groupby("compute").trunc.mean().mul(100).round(1).rename("% truncated (by compute)")], axis=1).T)
display(sel.groupby("correct", dropna=False).excluded_labeling.agg(["mean", "size"]).rename(columns={"mean": "share_not_labeled_valid", "size": "n"}))

## 5. Trace-level and segment-level feature preparation

Everything below uses the **eligible** traces (labeled, complete, correctness known). Prepared once and reused:
* `tr` – one row per trace (length, segment structure, role words/shares, role runs, answer events, overlap).
* `segE` – eligible segments in original order with `start/end` (cumulative words) and `rel_start/rel_end`.
* `runs` – consecutive same-role segments merged into **phase runs** (a role *change* = boundary between runs; this is not the same as the segment count).
* `A` / `ch` – answer-bearing segments and observed answer changes.

**Answer events (observed annotation events).** An answer-bearing segment has `answer_status` in {candidate, final} *and* a non-null `answer_value`. Values are normalised only lightly (lower-case, `\boxed{}`/`$`/commas/trailing period removed, numerics compared as floats). An *observed change* = a different normalised value than the previous answer-bearing segment. Changes are tagged `replacement` or `refinement_or_format` (one non-numeric value contained in the other). This is an algorithmic reading of annotations, not a verified judgement of what the model "meant".

In [ ]:
tr = df_t[df_t.eligible].copy(); tr.index = tr.trace_id.values; tr.index.name = None
segE = seg[seg.eligible].copy()
segE["nonempty"] = segE.words > 0
g = segE.groupby("trace_id")
tr["n_seg"] = g.size(); tr["n_nonempty"] = g.nonempty.sum()
tr["mean_seg_words"] = tr.total_words / tr.n_nonempty
tr["median_seg_words"] = segE[segE.nonempty].groupby("trace_id").words.median()
tr["max_seg_words"] = g.words.max()

rw = segE.pivot_table(index="trace_id", columns="role", values="words", aggfunc="sum", fill_value=0).reindex(columns=ROLES, fill_value=0)
rn = pd.crosstab(segE.trace_id, segE.role).reindex(columns=ROLES, fill_value=0)
for r in ROLES:
    tr[f"w_{r}"] = rw[r]; tr[f"p_{r}"] = 100 * rw[r] / tr.total_words; tr[f"n_{r}"] = rn[r]

# phase runs (consecutive segments with the same role merged; original segments stay in segE)
newrun = (segE.role != segE.role.shift()) | (segE.trace_id != segE.trace_id.shift())
runs = (segE.assign(run_id=newrun.cumsum()).groupby("run_id")
        .agg(trace_id=("trace_id", "first"), role=("role", "first"), seg_first=("seg_pos", "min"), seg_last=("seg_pos", "max"),
             n_seg=("seg_pos", "size"), words=("words", "sum"), start=("start", "min"), end=("end", "max")).reset_index(drop=True))
runs["run_idx"] = runs.groupby("trace_id").cumcount()
runs["total"] = runs.trace_id.map(tr.total_words)
runs["rel_start"] = runs.start / runs.total; runs["rel_end"] = runs.end / runs.total
gr = runs.groupby("trace_id")
runs["prev_role"] = gr.role.shift(1); runs["next_role"] = gr.role.shift(-1)
runs["next_words"] = gr.words.shift(-1); runs["next_seg_last"] = gr.seg_last.shift(-1)
tr["n_runs"] = gr.size(); tr["n_role_changes"] = tr.n_runs - 1
tr["chg_per_100w"] = 100 * tr.n_role_changes / tr.total_words
tr["chg_per_seg"] = tr.n_role_changes / (tr.n_seg - 1).where(tr.n_seg > 1)
for r in ROLES:
    sub = runs[runs.role == r]
    tr[f"nruns_{r}"] = sub.groupby("trace_id").size(); tr[f"nruns_{r}"] = tr[f"nruns_{r}"].fillna(0)
for r in KEY:
    sub = runs[runs.role == r].groupby("trace_id")
    tr[f"first_{r}_rel"] = sub.rel_start.min(); tr[f"first_{r}_word"] = sub.start.min()
runs_by = {k: v for k, v in runs.groupby("trace_id")}
seg_g = {k: v for k, v in segE.groupby("trace_id")}

In [ ]:
# ---- answer events ----
def norm_ans(x):
    if x is None or (isinstance(x, float) and np.isnan(x)): return np.nan
    t = str(x).strip().lower()
    t = re.sub(r"\\boxed\{(.*)\}", r"\1", t)
    t = re.sub(r"\\(text|mathrm|textbf)\{(.*?)\}", r"\2", t)
    t = t.replace("$", "").replace("\\!", "").replace(",", "").replace("\\%", "%")
    t = re.sub(r"\s+", " ", t).strip(" .")
    if t in ("", "none", "null", "nan"): return np.nan
    try: return f"{float(t.rstrip('%')):g}"
    except ValueError: return t

def isnum(t):
    try: float(t); return True
    except ValueError: return False

def chg_type(p, c):
    if isnum(p) and isnum(c): return "replacement"
    return "refinement_or_format" if (p in c or c in p) else "replacement"

def window_has(role, k=3):    # role present in this segment or the previous k-1 segments of the same trace
    x = segE.role == role; out = x.copy()
    for j in range(1, k):
        out |= x.shift(j, fill_value=False) & (segE.trace_id.shift(j) == segE.trace_id)
    return out
for r in KEY: segE[f"win_{r}"] = window_has(r)

tr["gold_norm"] = tr.gold.map(norm_ans)
segE["ans"] = segE.answer_value.map(norm_ans).where(segE.answer_status.isin(["candidate", "final"]))
A = segE[segE.ans.notna()].copy()
A["gold_norm"] = A.trace_id.map(tr.gold_norm)
A["prev_ans"] = A.groupby("trace_id").ans.shift()
A["is_change"] = A.prev_ans.notna() & (A.ans != A.prev_ans)
ch = A[A.is_change].copy()
ch["chg_type"] = [chg_type(p, c) for p, c in zip(ch.prev_ans, ch.ans)]
ch["direction"] = np.select([ch.ans == ch.gold_norm, ch.prev_ans == ch.gold_norm], ["toward_gold", "away_from_gold"], "between_non_gold")
A = A.join(ch[["chg_type", "direction"]])

gA = A.groupby("trace_id")
ans_tr = pd.DataFrame({"first_ans_val": gA.ans.first(), "first_ans_seg": gA.seg_pos.first(), "first_ans_words_before": gA.start.first(),
                       "first_ans_end": gA.end.first(), "first_ans_rel": gA.rel_start.first(), "last_ans": gA.ans.last(), "n_ans_values": gA.ans.nunique()})
F = A[A.answer_status == "final"]; gF = F.groupby("trace_id")
ans_tr["first_final_seg"] = gF.seg_pos.first(); ans_tr["first_final_end"] = gF.end.first()
ans_tr["first_final_rel"] = gF.rel_start.first(); ans_tr["first_final_words_before"] = gF.start.first()
ans_tr["n_final_values"] = gF.ans.nunique()
ans_tr["final_ans"] = gF.ans.last().reindex(ans_tr.index).fillna(ans_tr.last_ans)      # last 'final' value, else last candidate
gC = ch.groupby("trace_id")
ans_tr["n_ans_changes"] = gC.size(); ans_tr["n_replacements"] = ch[ch.chg_type == "replacement"].groupby("trace_id").size()
ans_tr[["n_ans_changes", "n_replacements"]] = ans_tr[["n_ans_changes", "n_replacements"]].fillna(0)
ans_tr["last_change_end"] = gC.end.last(); ans_tr["last_change_rel"] = gC.rel_start.last()
ans_tr["gold_seen"] = (A.ans == A.gold_norm).groupby(A.trace_id).any()
tr = tr.join(ans_tr)
tr["has_ans"] = tr.first_ans_val.notna(); tr["has_final"] = tr.first_final_seg.notna()
tr["stab_end"] = tr.last_change_end.fillna(tr.first_ans_end)          # answer stable from here on
tr["post_stab_words"] = tr.total_words - tr.stab_end; tr["post_stab_share"] = tr.post_stab_words / tr.total_words
tr["post_first_final_words"] = tr.total_words - tr.first_final_end
tr["agree_gold"] = np.where(tr.final_ans.notna() & tr.gold_norm.notna(), (tr.final_ans == tr.gold_norm).astype(float), np.nan)
tr["lost_gold"] = np.where(tr.has_ans, ((tr.gold_seen == True) & (tr.agree_gold == 0)).astype(float), np.nan)
segE["after_first_final"] = segE.seg_pos > segE.trace_id.map(tr.first_final_seg)
print(f"eligible traces with >=1 answer-bearing segment: {tr.has_ans.mean():.1%}; with a 'final': {tr.has_final.mean():.1%}; observed answer changes: {len(ch):,}")

### 5b. Lexical overlap features (repetition / new content)

Lightweight and interpretable: **TF-IDF cosine similarity** between segments of the same trace (word-level, fitted once on all eligible segments; numbers count as tokens). For each segment (≥ 5 words):
* `sim_prev`: max similarity to any earlier segment; `sim_nonadj`: max similarity to any earlier **non-adjacent** segment (skips the immediately preceding one, which is often a natural continuation);
* `novel_frac`: share of its distinct content words (small stop-word list removed) not seen earlier in the trace;
* `rep4_frac`: share of its word 4-grams that already occurred in an earlier segment.

High overlap is **not** evidence of unproductive text (verification and correction legitimately revisit earlier content), and novelty naturally decays with position as the vocabulary saturates, so comparisons are always made at matched relative positions or against the same question's other trace.

In [ ]:
MIN_SIM_WORDS = 5
STOP = set("the a an and or of to in is are was be it that this we i so then if for on as with by at from can let me my have has not but".split())
segE["xi"] = np.arange(len(segE))
segE["clean"] = segE.text.map(clean)
vec = TfidfVectorizer(lowercase=True, token_pattern=r"(?u)\b\w+\b", sublinear_tf=True)
X = vec.fit_transform(segE.clean)
sim_prev = np.full(len(segE), np.nan); sim_nonadj = np.full(len(segE), np.nan)
novel = np.full(len(segE), np.nan); rep4 = np.full(len(segE), np.nan); n4 = np.zeros(len(segE)); r4 = np.zeros(len(segE))
words_arr = segE.words.values
for tid, idx in segE.groupby("trace_id").indices.items():
    n = len(idx); Xt = X[idx]
    S = (Xt @ Xt.T).toarray(); ok = words_arr[idx] >= MIN_SIM_WORDS
    for off, out in [(-1, sim_prev), (-2, sim_nonadj)]:
        m = np.tril(np.ones((n, n), bool), off) & ok[None, :]
        v = np.where(m, S, -1.0).max(axis=1); v[(v < 0) | ~ok] = np.nan; out[idx] = v
    seen, seen4 = set(), set()
    for i in idx:
        toks = re.findall(r"\w+", segE.clean.iat[i].lower())
        content = {w for w in toks if w not in STOP}
        if len(content) >= 3: novel[i] = len(content - seen) / len(content)
        seen |= content
        g4 = [tuple(toks[j:j + 4]) for j in range(len(toks) - 3)]
        if g4:
            r4[i] = sum(x in seen4 for x in g4); n4[i] = len(g4); rep4[i] = r4[i] / n4[i]
        seen4.update(g4)
segE["sim_prev"], segE["sim_nonadj"], segE["novel_frac"], segE["rep4_frac"], segE["r4"], segE["n4"] = sim_prev, sim_nonadj, novel, rep4, r4, n4
gS = segE.groupby("trace_id")
tr["rep4_trace"] = gS.r4.sum() / gS.n4.sum().where(gS.n4.sum() > 0)
tr["mean_overlap"] = segE.assign(w=segE.words * segE.sim_nonadj.notna(), ws=segE.words * segE.sim_nonadj.fillna(0)).groupby("trace_id").apply(lambda x: x.ws.sum() / x.w.sum() if x.w.sum() > 0 else np.nan)
tr["share_words_overlap"] = segE.assign(h=segE.words * (segE.sim_nonadj >= 0.6)).groupby("trace_id").h.sum() / tr.total_words
# run-level: overlap of the first segment of each run
key = segE.set_index(["trace_id", "seg_pos"])
mi = pd.MultiIndex.from_arrays([runs.trace_id, runs.seg_first])
runs["first_sim"] = key.sim_nonadj.reindex(mi).values; runs["first_novel"] = key.novel_frac.reindex(mi).values
print("overlap features done:", f"{segE.sim_nonadj.notna().mean():.1%} of segments have a non-adjacent similarity (>= {MIN_SIM_WORDS} words with an earlier segment)")

### 5c. Reusable helpers (divergence detection, paired statistics)

**Meaningful structural divergence** between two comparable traces A and B is searched with three transparent detectors (each returns a position on A and on B, in absolute words):
* **D1 alignment** – `difflib` alignment of the two *merged-phase* role sequences; the first non-matching block that involves ≥ `DIV_MIN_WORDS` words (tiny blocks are ignored).
* **D2 first answer differs** – both traces state a first candidate and the normalised values differ.
* **D3 key-phase asymmetry** – a doubt/verification/correction phase first appears in only one trace, or at relative positions ≥ 25 percentage points apart.

The earliest detector (mean relative position) is the *algorithmically detected* candidate; it must be **read in the text** before being called a substantive divergence (`show_divergence`). Nothing here claims the divergence caused a difference in outcome.

In [ ]:
def first_run(tid, role):
    r = runs_by[tid]; m = r[r.role == role]
    return (m.start.iloc[0], m.rel_start.iloc[0]) if len(m) else (np.nan, np.nan)

def find_divergence(ta, tb):
    ra, rb = runs_by[ta], runs_by[tb]; Wa, Wb = tr.at[ta, "total_words"], tr.at[tb, "total_words"]
    det = {}
    sm = SequenceMatcher(a=ra.role.tolist(), b=rb.role.tolist(), autojunk=False)
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal": continue
        if ra.words.iloc[i1:i2].sum() + rb.words.iloc[j1:j2].sum() >= DIV_MIN_WORDS:
            wa = ra.start.iloc[i1] if i1 < len(ra) else Wa; wb = rb.start.iloc[j1] if j1 < len(rb) else Wb
            det["D1_alignment"] = (wa, wb, f"{tag}: A{ra.role.iloc[i1:i2].tolist()} vs B{rb.role.iloc[j1:j2].tolist()}"); break
    va, vb = tr.at[ta, "first_ans_val"], tr.at[tb, "first_ans_val"]
    if pd.notna(va) and pd.notna(vb) and va != vb:
        det["D2_first_answer_differs"] = (tr.at[ta, "first_ans_words_before"], tr.at[tb, "first_ans_words_before"], f"{va} vs {vb}")
    d3 = []
    for role in KEY:
        (wa, fa), (wb, fb) = first_run(ta, role), first_run(tb, role)
        if np.isnan(fa) and np.isnan(fb): continue
        if np.isnan(fa) != np.isnan(fb):
            rel = fb if np.isnan(fa) else fa
            d3.append((rel, rel * Wa if np.isnan(fa) else wa, rel * Wb if np.isnan(fb) else wb, f"{role} only in {'B' if np.isnan(fa) else 'A'}"))
        elif abs(fa - fb) >= 0.25:
            rel = min(fa, fb); d3.append((rel, rel * Wa, rel * Wb, f"{role} first at {fa:.0%} (A) vs {fb:.0%} (B)"))
    if d3:
        _, wa, wb, note = min(d3); det["D3_key_phase_asymmetry"] = (wa, wb, note)
    res = {"detectors": det, "earliest": None, "a_word": np.nan, "b_word": np.nan, "rel": np.nan}
    for name, (wa, wb, note) in det.items():
        rel = np.nanmean([wa / Wa, wb / Wb])
        det[name] = (wa, wb, note, rel)
        if res["earliest"] is None or rel < res["rel"]: res.update(earliest=name, a_word=wa, b_word=wb, rel=rel)
    return res

def show_divergence(ta, tb, res, k=2):
    if res["earliest"] is None: return pd.DataFrame()
    out = []
    for lab, tid, w in [("A", ta, res["a_word"]), ("B", tb, res["b_word"])]:
        s = seg_g[tid]; c = min(np.searchsorted(s.end.values, w, side="right"), len(s) - 1)
        for i in range(max(0, c - k), min(len(s), c + k + 1)):
            r = s.iloc[i]
            out.append(dict(trace=lab, at_divergence="<<" if i == c else "", seg_pos=r.seg_pos, role=r.role, words=r.words, status=r.answer_status,
                            value=r.answer_value, temporal=r.temporal_role, text=r.text.strip().replace("\n", " ")[:200]))
    return pd.DataFrame(out)

def boot_ci(x, stat=np.median, n=2000):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 3: return (np.nan, np.nan)
    return tuple(np.percentile(stat(x[rng.integers(0, len(x), (n, len(x)))], axis=1), [2.5, 97.5]))

def paired_summary(wide, feats, a, b):
    # wide has columns f"{feat}__{a}" and f"{feat}__{b}", one row per question (or pair); question-level paired differences a - b
    rows = []
    for f in feats:
        dd = (wide[f"{f}__{a}"] - wide[f"{f}__{b}"]).dropna(); lo, hi = boot_ci(dd)
        p = stats.wilcoxon(dd).pvalue if len(dd) >= 6 and (dd != 0).any() else np.nan
        rows.append(dict(feature=f, n=len(dd), **{f"median_{a}": wide[f"{f}__{a}"].median(), f"median_{b}": wide[f"{f}__{b}"].median()},
                         median_diff=dd.median(), ci_lo=lo, ci_hi=hi, frac_a_greater=(dd > 0).mean(), wilcoxon_p=p))
    return pd.DataFrame(rows).set_index("feature")

def paired_wide(d, feats, col="correct", a=1, b=0):    # question-level means within each side (rows of d should be matched cells)
    m = d.groupby(["qkey", col])[feats].mean().unstack(col)
    m.columns = [f"{f}__{'a' if c == a else 'b'}" for f, c in m.columns]
    return m.dropna(how="any")

def wide_from_pairs(ia, ib, feats):
    wa, wb = tr.loc[list(ia), feats].reset_index(drop=True), tr.loc[list(ib), feats].reset_index(drop=True)
    return pd.concat([wa.add_suffix("__a"), wb.add_suffix("__b")], axis=1)

ROLE_W = [f"w_{r}" for r in ROLES]; ROLE_P = [f"p_{r}" for r in ROLES]
FEATS = (["total_words", "n_seg", "n_nonempty", "mean_seg_words", "n_runs", "n_role_changes", "chg_per_100w"] + ROLE_W + ROLE_P +
         ["nruns_doubt", "nruns_verification", "nruns_correction", "first_ans_words_before", "first_ans_rel", "post_stab_words",
          "post_stab_share", "n_ans_changes", "rep4_trace", "mean_overlap"])

## 6. Exploratory analysis of reasoning length and phase structure

Do longer traces have more segments, longer segments, or both? Spearman correlations are computed **within question × compute-level cells** (≥ 4 eligible samples) so that between-question difficulty differences do not dominate. Truncated traces are summarised separately.

In [ ]:
tr["len_q"] = pd.cut(tr.groupby(["benchmark", "compute"]).total_words.rank(pct=True), [0, .25, .5, .75, 1.0],
                     labels=["Q1 shortest", "Q2", "Q3", "Q4 longest"])
def cell_spearman(d, x, y, min_n=4):
    out = []
    for _, g_ in d.groupby(["qkey", "compute"]):
        if len(g_) >= min_n and g_[x].nunique() > 1 and g_[y].nunique() > 1: out.append(stats.spearmanr(g_[x], g_[y])[0])
    return np.array(out)
corr_rows = []
for x, y in [("total_words", "n_nonempty"), ("total_words", "mean_seg_words"), ("n_nonempty", "mean_seg_words"),
             ("total_words", "n_role_changes"), ("n_seg", "n_role_changes"), ("total_words", "thinking_tokens")]:
    v = cell_spearman(tr, x, y)
    corr_rows.append(dict(x=x, y=y, n_cells=len(v), median_rho=np.nanmedian(v) if len(v) else np.nan, share_positive=(v > 0).mean() if len(v) else np.nan))
corr_tab = pd.DataFrame(corr_rows); display(corr_tab); save_tab(corr_tab, "length_structure_correlations", index=False)
R["corr"] = corr_tab.set_index(["x", "y"])

wpt = (tr.thinking_tokens / tr.total_words).replace([np.inf, -np.inf], np.nan)
print(f"thinking tokens per word (supplementary): median {wpt.median():.2f}, IQR {wpt.quantile(.25):.2f}-{wpt.quantile(.75):.2f} -> tokens and words are related but not interchangeable")

fig, ax = plt.subplots(1, 4, figsize=(21, 4.6))
sns.scatterplot(data=tr, x="total_words", y="n_nonempty", hue="compute", alpha=.45, s=18, ax=ax[0]); ax[0].set(xscale="log", yscale="log", title="Segments vs words")
sns.scatterplot(data=tr, x="total_words", y="mean_seg_words", hue="compute", alpha=.45, s=18, ax=ax[1], legend=False); ax[1].set(xscale="log", yscale="log", title="Mean segment size vs words")
q = tr.groupby("len_q", observed=True)[ROLE_W].mean(); q.columns = ROLES
q.plot(kind="bar", stacked=True, ax=ax[2], color=[ROLE_COLORS[r] for r in ROLES], legend=False); ax[2].set(title="Mean words per role by length quartile\n(quartile within benchmark x compute)", ylabel="words", xlabel="")
qp = tr.groupby("len_q", observed=True)[ROLE_P].mean(); qp.columns = ROLES
qp.plot(kind="bar", stacked=True, ax=ax[3], color=[ROLE_COLORS[r] for r in ROLES]); ax[3].set(title="Mean share of trace by role (%)", ylabel="% of words", xlabel="")
ax[3].legend(loc="center left", bbox_to_anchor=(1, .5), title="role")
for a_ in ax[2:]: a_.tick_params(axis="x", rotation=20)
plt.tight_layout(); save_fig("fig01_length_structure"); plt.show()
display(pd.concat([q.add_prefix("words_"), qp.add_prefix("pct_")], axis=1).round(1))

tru = df_t[df_t.trunc == 1]
print(f"\nTruncated traces: {len(tru):,} ({len(tru)/len(df_t):.1%}); with valid labels: {int(tru.labeled_ok.sum()):,}; median words {tru.total_words.median():.0f} (eligible traces: {tr.total_words.median():.0f})")
display(pd.DataFrame({"eligible_(complete)": tr.groupby("compute").total_words.median(), "truncated": tru.groupby("compute").total_words.median(),
                      "n_truncated": tru.groupby("compute").size()}).round(0))
display(tr.groupby(["compute", "correct"]).agg(n=("total_words", "size"), median_words=("total_words", "median"), median_segments=("n_nonempty", "median"),
                                               median_seg_words=("median_seg_words", "median")).round(1))

## 7. Question groups and Group 1: all-correct questions with very different reasoning lengths

**Eligibility.** Only eligible traces count. A question needs at least `MIN_SAMPLES` eligible samples (automatically lowered, down to 2, if any group would otherwise contain fewer than 3 questions; the value used is printed).
* **Group 1** – every eligible sample correct **and** `max − min ≥ 1.5·SD` **and** `max/min ≥ 1.5` (the second condition is added because with few samples the SD rule alone is nearly always met).
* **Group 2** – at least one correct and one incorrect eligible sample.
* **Group 3** – every eligible sample incorrect.

Pairs: the shortest/longest trace of a Group-1 question are taken **within the same compute level** when some compute level has ≥ 2 samples (flag `same_compute`), otherwise across levels – compute budget is a confound for length.

In [ ]:
Q = tr.groupby("qkey").agg(benchmark=("benchmark", "first"), difficulty=("difficulty_tier", "first"), n=("total_words", "size"),
                           n_correct=("correct", "sum"), min_w=("total_words", "min"), max_w=("total_words", "max"),
                           med_w=("total_words", "median"), sd_w=("total_words", "std"), n_compute=("compute", "nunique"))
Q["range_w"] = Q.max_w - Q.min_w; Q["ratio_w"] = Q.max_w / Q.min_w

def assign_groups(Q, min_n):
    ok = Q.n >= min_n; allc = ok & (Q.n_correct == Q.n); allw = ok & (Q.n_correct == 0); mixed = ok & ~allc & ~allw
    g1 = allc & (Q.range_w >= LEN_RANGE_SD_K * Q.sd_w) & (Q.ratio_w >= LEN_MIN_RATIO)
    return pd.Series(np.select([g1, mixed, allw, allc, ~ok], ["G1", "G2", "G3", "all_correct_low_variability", "too_few_samples"], "other"), index=Q.index)

for min_n in range(MIN_SAMPLES, 1, -1):
    Q["group"] = assign_groups(Q, min_n); cnt = Q.group.value_counts()
    if all(cnt.get(k, 0) >= 3 for k in ["G1", "G2", "G3"]): break
R["min_n_used"] = min_n
print(f"minimum eligible samples per question used: {min_n} (configured: {MIN_SAMPLES})")
tr["group"] = tr.qkey.map(Q.group)
tr["cond"] = np.select([tr.group == "G1", (tr.group == "G2") & (tr.correct == 1), (tr.group == "G2") & (tr.correct == 0), tr.group == "G3"],
                       ["G1 correct", "G2 correct", "G2 incorrect", "G3 incorrect"], "other")
for d_ in (runs, segE, A, ch):
    d_["cond"] = d_.trace_id.map(tr.cond); d_["qkey"] = d_.trace_id.map(tr.qkey); d_["compute"] = d_.trace_id.map(tr.compute); d_["correct"] = d_.trace_id.map(tr.correct)

allq = df_t.groupby("qkey").agg(n_all=("trace_id", "size"), n_truncated=("trunc", lambda x: int((x == 1).sum())), n_not_labeled=("labeled_ok", lambda x: int((~x).sum())))
gs = Q.join(allq).groupby("group").agg(questions=("n", "size"), eligible_traces=("n", "sum"), all_traces_of_these_questions=("n_all", "sum"),
                                      truncated_excluded=("n_truncated", "sum"), not_labeled_excluded=("n_not_labeled", "sum"),
                                      median_samples=("n", "median"), benchmarks=("benchmark", "nunique"))
display(gs); save_tab(gs, "group_selection")
display(pd.crosstab(Q.benchmark, Q.group)); display(pd.crosstab(Q.difficulty, Q.group))
G1q, G2q, G3q = [Q.index[Q.group == k] for k in ("G1", "G2", "G3")]
for k, v in zip(("G1", "G2", "G3"), (G1q, G2q, G3q)):
    R[f"{k}_n"] = len(v)
    if len(v) == 0: print(f"WARNING: no questions satisfy the {k} criteria with this dataset; related sections will be skipped.")

In [ ]:
def pick_pair(g_):
    cells = [c for _, c in g_.groupby("compute") if len(c) >= 2]
    if cells: c, same = max(cells, key=lambda c: c.total_words.max() - c.total_words.min()), True
    else: c, same = g_, False
    return c.total_words.idxmin(), c.total_words.idxmax(), same

pairs1 = pd.DataFrame()
if len(G1q):
    pr = [(qk, *pick_pair(tr[tr.qkey == qk])) for qk in G1q]
    pairs1 = pd.DataFrame(pr, columns=["qkey", "short", "long", "same_compute"])
    ps, pl = tr.loc[pairs1.short, FEATS].reset_index(drop=True), tr.loc[pairs1.long, FEATS].reset_index(drop=True)
    dlt = pl - ps
    pairs1["benchmark"] = pairs1.qkey.map(Q.benchmark); pairs1["compute_short"] = tr.loc[pairs1.short, "compute"].values; pairs1["compute_long"] = tr.loc[pairs1.long, "compute"].values
    pairs1["words_short"], pairs1["words_long"] = ps.total_words.values, pl.total_words.values
    pairs1["range"] = pairs1.words_long - pairs1.words_short
    extra = dlt.total_words.values
    shares = dlt[ROLE_W].div(dlt.total_words, axis=0); shares.columns = ROLES
    lr = np.log(pl.total_words / ps.total_words).where(lambda x: x > 0.05)
    pairs1["frac_len_ratio_from_segment_count"] = (np.log(pl.n_nonempty / ps.n_nonempty) / lr).values       # log(W) = log(n_seg) + log(mean seg size)
    pre_s, pre_l = ps.first_ans_words_before, pl.first_ans_words_before
    pairs1["share_extra_before_first_answer"] = ((pre_l - pre_s) / dlt.total_words).values
    pairs1["first_ans_rel_short"], pairs1["first_ans_rel_long"] = ps.first_ans_rel.values, pl.first_ans_rel.values
    pairs1["post_stab_short"], pairs1["post_stab_long"] = ps.post_stab_words.values, pl.post_stab_words.values
    pairs1["changes_short"], pairs1["changes_long"] = ps.n_ans_changes.values, pl.n_ans_changes.values
    pairs1["same_final_answer"] = (tr.loc[pairs1.short, "final_ans"].values == tr.loc[pairs1.long, "final_ans"].values)
    for r in ROLES: pairs1[f"extra_{r}"] = dlt[f"w_{r}"].values; pairs1[f"extra_share_{r}"] = shares[r].values
    for c in ["n_nonempty", "mean_seg_words", "n_runs"]: pairs1[f"{c}_short"], pairs1[f"{c}_long"] = ps[c].values, pl[c].values
    for r in ROLES: pairs1[f"pct_{r}_short"], pairs1[f"pct_{r}_long"] = ps[f"p_{r}"].values, pl[f"p_{r}"].values
    save_tab(pairs1, "group1_pairs", index=False)

    print(f"Group 1: {len(pairs1)} questions; pair within the same compute level for {pairs1.same_compute.sum()}; "
          f"median words short/long = {pairs1.words_short.median():.0f}/{pairs1.words_long.median():.0f} (long = {np.median(pairs1.words_long/pairs1.words_short):.2f}x)")
    rows = []
    for r in ROLES:
        lo, hi = boot_ci(shares[r])
        rows.append(dict(role=r, median_extra_words=pairs1[f"extra_{r}"].median(), median_share_of_extra=shares[r].median(), ci_lo=lo, ci_hi=hi,
                         median_pct_short=ps[f"p_{r}"].median(), median_pct_long=pl[f"p_{r}"].median(),
                         share_of_pairs_where_long_has_higher_pct=(pl[f"p_{r}"] > ps[f"p_{r}"]).mean()))
    g1role = pd.DataFrame(rows).set_index("role"); display(g1role.round(3)); save_tab(g1role, "group1_extra_words_by_role")
    R["g1_role"] = g1role
    fc = pairs1.frac_len_ratio_from_segment_count.dropna()
    both = ((pl.n_nonempty > ps.n_nonempty) & (pl.mean_seg_words > ps.mean_seg_words)).mean()
    print(f"Length ratio explained by MORE segments (vs LARGER segments): median {fc.median():.0%} of the log-ratio comes from segment count; "
          f"long trace has both more and longer segments in {both:.0%} of pairs")
    R["g1_frac_count"], R["g1_both"] = fc.median(), both
    has = pairs1.share_extra_before_first_answer.notna()
    print(f"Where does the extra text sit? (answer annotation available for {has.sum()}/{len(pairs1)} pairs) median share of extra words located BEFORE the first answer: "
          f"{pairs1.share_extra_before_first_answer.median():.0%}; first answer at {np.nanmedian(ps.first_ans_rel):.0%} (short) vs {np.nanmedian(pl.first_ans_rel):.0%} (long) of the trace")
    R["g1_pre_share"] = pairs1.share_extra_before_first_answer.median(); R["g1_first_rel"] = (np.nanmedian(ps.first_ans_rel), np.nanmedian(pl.first_ans_rel))
    g1sum = paired_summary(pd.concat([pl.add_suffix("__l"), ps.add_suffix("__s")], axis=1),
                           ["n_nonempty", "mean_seg_words", "n_runs", "n_role_changes", "chg_per_100w"] + ROLE_P + ["first_ans_rel", "post_stab_words", "n_ans_changes"], "l", "s")
    display(g1sum.round(3)); save_tab(g1sum, "group1_long_minus_short_paired"); R["g1_sum"] = g1sum

    fig, ax = plt.subplots(1, 3, figsize=(18, 4.6))
    sns.boxplot(data=pairs1.melt(value_vars=[f"extra_{r}" for r in ROLES], var_name="role", value_name="extra_words").assign(role=lambda x: x.role.str[6:]),
                x="role", y="extra_words", order=ROLES, palette=ROLE_COLORS, ax=ax[0]); ax[0].axhline(0, c="k", lw=.7); ax[0].set(title="Extra words in long vs short trace, by role\n(one point-pair per Group-1 question)", ylabel="long - short (words)")
    sns.histplot(fc.clip(-0.5, 1.5), bins=15, ax=ax[1], color="#4c78a8"); ax[1].set(title="Share of the length ratio due to MORE segments\n(rest: larger segments)", xlabel="fraction (clipped to [-0.5,1.5])")
    sns.histplot(pairs1.share_extra_before_first_answer.dropna().clip(-0.5, 1.5), bins=15, ax=ax[2], color="#54a24b"); ax[2].set(title="Share of extra words before the first answer appears", xlabel="fraction (clipped)")
    plt.tight_layout(); save_fig("fig02_group1_extra_words"); plt.show()

## 8. Group 2: mixed-correctness questions

Correct and incorrect attempts are compared **within question**, using only **matched compute cells** (question × compute level containing both outcomes), so the budget is held fixed. Two views:
1. **All matched cells**: question-level mean of each feature for correct vs incorrect samples, paired across questions.
2. **Length-matched pairs**: one pair per question (the correct/incorrect pair with the most similar word count in a shared compute level); pairs with word ratio ≤ 1.25 are analysed separately so that *structure* is separated from *length*. Role shares, change rates per 100 words and per-segment sizes are length-normalised features.

Question-level paired differences (median, bootstrap CI over questions, Wilcoxon p) are reported as descriptive/exploratory evidence; no multiplicity correction is applied.

In [ ]:
pairs2 = pd.DataFrame(); trm = pd.DataFrame()
if len(G2q):
    c2 = tr[tr.qkey.isin(G2q)]
    flag = c2.groupby(["qkey", "compute"]).correct.transform(lambda x: bool(x.min() == 0 and x.max() == 1))
    trm = c2[flag.astype(bool)]
    print(f"Group 2: {len(G2q)} questions; {trm.qkey.nunique()} have >=1 matched compute cell ({len(trm)} traces); "
          f"{len(G2q) - trm.qkey.nunique()} questions only mix outcomes across compute levels (excluded from the matched analysis)")
    R["g2_matched_n"] = trm.qkey.nunique()
    pr = []
    for qk, g_ in trm.groupby("qkey"):
        best = None
        for cl, c in g_.groupby("compute"):
            for tc in c.index[c.correct == 1]:
                for ti in c.index[c.correct == 0]:
                    lr_ = abs(np.log(c.total_words[tc] / c.total_words[ti]))
                    if best is None or lr_ < best[0]: best = (lr_, tc, ti, cl)
        pr.append(dict(qkey=qk, tc=best[1], ti=best[2], compute=best[3], len_ratio=float(np.exp(best[0]))))
    pairs2 = pd.DataFrame(pr)
    pairs2["words_c"], pairs2["words_i"] = tr.loc[pairs2.tc, "total_words"].values, tr.loc[pairs2.ti, "total_words"].values
    pairs2["n_q"] = pairs2.qkey.map(Q.n)
    save_tab(pairs2, "group2_length_matched_pairs", index=False)
    F2 = FEATS + ["has_ans"]
    wide_all = paired_wide(trm.assign(has_ans=trm.has_ans.astype(float)), F2)
    s_all = paired_summary(wide_all, F2, "a", "b").rename(columns={"median_a": "median_correct", "median_b": "median_incorrect", "frac_a_greater": "frac_correct_greater"})
    lm = pairs2[pairs2.len_ratio <= 1.25]
    wide_lm = wide_from_pairs(lm.tc, lm.ti, FEATS)
    s_lm = paired_summary(wide_lm, FEATS, "a", "b").rename(columns={"median_a": "median_correct", "median_b": "median_incorrect", "frac_a_greater": "frac_correct_greater"})
    print(f"\n(a) ALL matched cells, question-level paired differences (correct - incorrect), n questions = {len(wide_all)}")
    display(s_all.round(3)); print(f"(b) LENGTH-MATCHED pairs (word ratio <= 1.25), n questions = {len(lm)}"); display(s_lm.round(3))
    save_tab(s_all, "group2_paired_all_matched"); save_tab(s_lm, "group2_paired_length_matched")
    R["g2_all"], R["g2_lm"], R["g2_lm_n"] = s_all, s_lm, len(lm)

    fig, ax = plt.subplots(1, 2, figsize=(17, 5.2))
    o = wide_all.assign(r=lambda x: x["total_words__b"] / x["total_words__a"]).sort_values("r")
    ax[0].hlines(range(len(o)), o["total_words__a"], o["total_words__b"], color="grey", lw=1)
    ax[0].scatter(o["total_words__a"], range(len(o)), c=COND_COLORS["G2 correct"], label="correct (mean)", s=22)
    ax[0].scatter(o["total_words__b"], range(len(o)), c=COND_COLORS["G2 incorrect"], label="incorrect (mean)", s=22)
    ax[0].set(xscale="log", xlabel="words (log scale)", ylabel="question (sorted by incorrect/correct ratio)", title="Group 2: length of correct vs incorrect attempts\n(matched compute cells)"); ax[0].legend()
    fr = []
    for lab, s_ in [("all matched cells", s_all), ("length-matched pairs", s_lm)]:
        for r in ROLES: fr.append(dict(role=r, analysis=lab, **s_.loc[f"p_{r}", ["median_diff", "ci_lo", "ci_hi"]].to_dict()))
    fr = pd.DataFrame(fr)
    for k, (lab, d_) in enumerate(fr.groupby("analysis")):
        y = np.arange(len(ROLES)) + (k - .5) * .3
        ax[1].errorbar(d_.median_diff, y, xerr=[d_.median_diff - d_.ci_lo, d_.ci_hi - d_.median_diff], fmt="o", capsize=3, label=lab)
    ax[1].axvline(0, c="k", lw=.7); ax[1].set_yticks(range(len(ROLES))); ax[1].set_yticklabels(ROLES)
    ax[1].set(xlabel="correct - incorrect, share of trace (percentage points)", title="Role shares: median question-level difference, 95% bootstrap CI"); ax[1].legend()
    plt.tight_layout(); save_fig("fig03_group2_length_and_role_shares"); plt.show()
else:
    print("Group 2 skipped: no eligible questions.")

## 9. Group 3: all-wrong questions

Questions of *recurring failure*. Do unsuccessful attempts follow similar trajectories? Structural similarity = `difflib` ratio between the merged-phase role sequences of two attempts (1 = identical sequence; at most 10 attempts per question are sampled for pairwise comparison). Answer behaviour is read from the annotation: whether attempts converge on the same final answer, change answers, or abandon a candidate matching `gold` (gold-matching uses the same light normalisation and may miss formatting differences).

In [ ]:
g3pairs = pd.DataFrame(); g3q = pd.DataFrame()
if len(G3q):
    c3 = tr[tr.qkey.isin(G3q)]; pr = []
    for qk, g_ in c3.groupby("qkey"):
        tids = list(g_.index) if len(g_) <= 10 else list(rng.choice(g_.index, 10, replace=False))
        seqs = {t: runs_by[t].role.tolist() for t in tids}
        for a_, b_ in itertools.combinations(tids, 2):
            wa, wb = tr.at[a_, "total_words"], tr.at[b_, "total_words"]
            pr.append(dict(qkey=qk, a=int(a_), b=int(b_), struct_sim=SequenceMatcher(a=seqs[a_], b=seqs[b_], autojunk=False).ratio(),
                           len_ratio=max(wa, wb) / min(wa, wb), same_compute=tr.at[a_, "compute"] == tr.at[b_, "compute"]))
    g3pairs = pd.DataFrame(pr)
    def g3_summary(qk, g_):
        fa = g_.final_ans.dropna(); vc = fa.value_counts(normalize=True)
        return pd.Series(dict(n=len(g_), median_words=g_.total_words.median(), cv_words=g_.total_words.std() / g_.total_words.mean(),
                              range_ratio=g_.total_words.max() / g_.total_words.min(), share_with_answer=g_.has_ans.mean(),
                              n_distinct_final=fa.nunique(), modal_final_share=vc.iloc[0] if len(vc) else np.nan,
                              mean_ans_changes=g_.n_ans_changes.mean(), share_lost_gold=g_.lost_gold.mean(),
                              mean_struct_sim=g3pairs.loc[g3pairs.qkey == qk, "struct_sim"].mean()))
    g3q = pd.DataFrame({qk: g3_summary(qk, g_) for qk, g_ in c3.groupby("qkey")}).T
    print(f"Group 3: {len(G3q)} questions, {len(c3)} traces")
    display(g3q.describe().loc[["25%", "50%", "75%"]].round(2)); save_tab(g3q, "group3_question_summary")
    conv = g3q[(g3q.n_distinct_final.notna()) & (g3q.share_with_answer * g3q.n >= 2)]
    print(f"Questions where >= half of attempts that state an answer give the SAME wrong answer: {(conv.modal_final_share >= .5).mean():.0%} of {len(conv)}; "
          f"median pairwise structural similarity {g3pairs.struct_sim.median():.2f}; median within-question max/min length ratio {g3q.range_ratio.median():.2f}")
    R["g3_conv"], R["g3_sim"], R["g3_rr"], R["g3_modal"] = (conv.modal_final_share >= .5).mean(), g3pairs.struct_sim.median(), g3q.range_ratio.median(), conv.modal_final_share.median()
    c3 = c3.assign(ans_pattern=np.select([~c3.has_ans, c3.lost_gold == 1, c3.n_ans_changes > 0], ["no answer annotated", "gold-matching candidate later replaced", "answer changed (never to gold)"], "single answer, never changed"))
    patt = c3.ans_pattern.value_counts(normalize=True).mul(100).round(1); display(patt.to_frame("% of Group-3 traces")); R["g3_pattern"] = patt

    fig, ax = plt.subplots(1, 3, figsize=(20, 5.5))
    top = g3q.sort_values("n", ascending=False).head(25).index; o = g3q.loc[top].sort_values("median_words").index.tolist()
    sns.stripplot(data=c3[c3.qkey.isin(top)], x="total_words", y="qkey", hue="compute", order=o, ax=ax[0], size=5, jitter=.15)
    ax[0].set(xscale="log", title="Group 3: word counts per question\n(25 questions with most samples)", xlabel="words (log)", ylabel=""); ax[0].set_yticklabels([s.split("::")[1][:14] for s in o], fontsize=7)
    sns.scatterplot(data=g3pairs, x="len_ratio", y="struct_sim", hue="same_compute", alpha=.5, ax=ax[1]); ax[1].set(xscale="log", title="Pairs of wrong attempts at one question:\nstructure vs length ratio", xlabel="word ratio (long/short)", ylabel="role-sequence similarity")
    patt.sort_values().plot(kind="barh", ax=ax[2], color="#7f3c8d"); ax[2].set(title="Answer behaviour in Group-3 traces", xlabel="% traces")
    plt.tight_layout(); save_fig("fig04_group3_overview"); plt.show()
else:
    print("Group 3 skipped: no eligible questions.")

## 10. Full-length absolute reasoning trajectories (essential figures)

Each trace is one horizontal row; **every segment is a block in its original order whose width equals its word count** and whose left edge is its cumulative word position. All rows in a figure share one x-axis (words), so a 500-word trace is five times as long as a 100-word trace. Zero-word segments get **no width** (a dotted tick marks their position). Nothing is sorted, merged, or rescaled.

Markers above the rows are **observed annotation events**: ▼ first answer-bearing segment (candidate or final), ★ first `final`, ◆ observed answer change. A dashed black line marks the **algorithmically detected** earliest divergence candidate between the two traces (sec. 12); it has not been verified by reading the text. Text of the segments is not drawn in the blocks; the companion tables below each figure give role-run summaries and the text of doubt/verification/correction segments.

Examples are chosen transparently: **G1** – the questions with the largest same-compute length range; **G2** – matched-compute questions with the most samples (length-matched pairs preferred); **G3** – questions with the most samples, showing a similar-length/least-similar-structure pair and the most different-length pair.

In [ ]:
chk = segE.groupby("trace_id").agg(ssum=("words", "sum"), last_end=("end", "max"), mono=("start", lambda x: x.is_monotonic_increasing))
assert (chk.ssum == chk.last_end).all() and chk.mono.all(), "segment widths must add up to the trace length"
assert segE.groupby("trace_id").seg_pos.apply(lambda x: x.is_monotonic_increasing).all(), "original order must be preserved"

seg_g = {k: v for k, v in segE.groupby("trace_id")}   # rebuilt now that overlap/cond columns exist
A_g = {k: v for k, v in A.groupby("trace_id")}
AB = {"setup": "S", "reasoning": "R", "doubt": "D", "verification": "V", "correction": "C"}

def tlabel(tid):
    r = tr.loc[tid]
    return f"{str(r.question_id)[:18]} | s{r.sample_index} | {'CORRECT' if r.correct == 1 else 'INCORRECT'} | compute {r.compute} | {int(r.total_words)}w, {int(r.n_nonempty)} seg"

def plot_traj(tids, title, normalized=False, divs=None, fname=None):
    n = len(tids); fig, ax = plt.subplots(figsize=(15, 0.95 * n + 2.4))
    xmax = 100 if normalized else max(tr.at[t, "total_words"] for t in tids)
    for y, tid in enumerate(tids):
        s = seg_g[tid]; k = 100 / tr.at[tid, "total_words"] if normalized else 1.0
        for st, w, role in zip(s.start, s.words, s.role):
            if w > 0: ax.broken_barh([(st * k, w * k)], (y - .36, .72), facecolors=ROLE_COLORS[role], edgecolors="white", linewidth=.5)
            else: ax.plot([st * k] * 2, [y - .36, y + .36], color="black", lw=.8, ls=":")
        a = A_g.get(tid)
        if a is not None:
            ax.plot(a.start.iloc[0] * k, y - .52, "v", color="black", ms=7)
            fin = a[a.answer_status == "final"]
            if len(fin): ax.plot(fin.start.iloc[0] * k, y - .52, "*", color="gold", mec="black", ms=12)
            c = a[a.is_change]
            if len(c): ax.plot(c.start * k, [y - .52] * len(c), "D", color="#d6404e", mec="black", ms=6)
        if divs and tid in divs and pd.notna(divs[tid]):
            ax.plot([divs[tid] * k] * 2, [y - .45, y + .45], color="black", lw=1.8, ls="--")
    ax.set_yticks(range(n)); ax.set_yticklabels([tlabel(t) for t in tids], fontsize=8)
    ax.set_ylim(n - .4, -.95); ax.set_xlim(0, xmax * 1.01)
    ax.set_xlabel("relative position (% of this trace's own words) - each row rescaled independently" if normalized
                  else "cumulative word position (absolute, shared scale across rows)")
    ax.set_title(title, fontsize=11); ax.grid(axis="y", visible=False)
    h = [Patch(color=ROLE_COLORS[r], label=r) for r in ROLES] + [
        Line2D([], [], marker="v", color="k", ls="", label="first answer"), Line2D([], [], marker="*", color="gold", mec="k", ls="", ms=11, label="first final"),
        Line2D([], [], marker="D", color="#d6404e", mec="k", ls="", label="answer change"), Line2D([], [], color="k", ls="--", label="detected divergence candidate"),
        Line2D([], [], color="k", ls=":", label="0-word segment (no width)")]
    ax.legend(handles=h, loc="upper center", bbox_to_anchor=(.5, -.18 if n < 3 else -.08), ncol=5, fontsize=8, frameon=False)
    plt.tight_layout()
    if fname: save_fig(fname)
    plt.show()

def seq_str(tid, maxruns=36):
    r = runs_by[tid]; s = " ".join(f"{AB[a]}{w}" for a, w in list(zip(r.role, r.words))[:maxruns])
    return s + (" ..." if len(r) > maxruns else "")

def dvc_text(tid, n=6):
    s = seg_g[tid]; s = s[s.role.isin(KEY)].head(n)
    return s.assign(text=s.text.str.strip().str.replace("\n", " ").str[:170])[["seg_pos", "role", "words", "rel_start", "answer_status", "answer_value", "temporal_role", "text"]].round(2)

EX = []
if len(pairs1):
    for _, r in pairs1.sort_values(["same_compute", "range"], ascending=False).head(N_EXAMPLES).iterrows():
        res = find_divergence(int(r.short), int(r.long))
        EX.append(dict(group="G1", tids=[int(r.short), int(r.long)], divs={int(r.short): res["a_word"], int(r.long): res["b_word"]},
                       title=f"GROUP 1 - same question, both correct: shortest (top) vs longest (bottom)  [{r.qkey}]"))
if len(pairs2):
    for _, r in pairs2.assign(lm=pairs2.len_ratio <= 1.25).sort_values(["lm", "n_q"], ascending=False).head(N_EXAMPLES).iterrows():
        res = find_divergence(int(r.tc), int(r.ti))
        EX.append(dict(group="G2", tids=[int(r.tc), int(r.ti)], divs={int(r.tc): res["a_word"], int(r.ti): res["b_word"]},
                       title=f"GROUP 2 - same question & compute level: correct (top) vs incorrect (bottom), length ratio {r.len_ratio:.2f}  [{r.qkey}]"))
if len(g3pairs):
    for qk in Q[Q.group == "G3"].sort_values("n", ascending=False).head(N_EXAMPLES).index:
        p = g3pairs[g3pairs.qkey == qk]; sel_ = []
        sim_len = p[p.len_ratio <= 1.25]
        if len(sim_len): r = sim_len.sort_values("struct_sim").iloc[0]; sel_ += [int(r.a), int(r.b)]
        r = p.sort_values("len_ratio", ascending=False).iloc[0]; sel_ += [int(r.a), int(r.b)]
        EX.append(dict(group="G3", tids=list(dict.fromkeys(sel_)), divs=None,
                       title=f"GROUP 3 - all attempts wrong: similar-length pair with least similar structure, then most different-length pair  [{qk}]"))

for i, ex in enumerate(EX):
    plot_traj(ex["tids"], "ABSOLUTE LENGTH | " + ex["title"], divs=ex["divs"], fname=f"fig05_abs_{ex['group']}_{i}")
    for t in ex["tids"]: print(f"  {tlabel(t)}\n    runs (role+words): {seq_str(t)}")
for grp in ("G1", "G2"):
    e = next((e for e in EX if e["group"] == grp), None)
    if e is not None:
        for t in e["tids"][::-1]:
            print(f"\nText of the first doubt/verification/correction segments - {tlabel(t)}"); display(dvc_text(t))

## 11. Normalized trajectories and phase-transition dynamics

### 11a. Normalized trajectories (same traces, each rescaled to 0–100 %)
These show *relative* organization only; they **cannot** be compared with the absolute maps for length.

In [ ]:
for i, ex in enumerate(EX):
    plot_traj(ex["tids"], "NORMALIZED (0-100% of each trace) | " + ex["title"], normalized=True, divs=ex["divs"], fname=f"fig06_norm_{ex['group']}_{i}")

### 11b. Role composition along the trace
For each trace, the share of every relative-position decile occupied by each role is computed exactly from the segment boundaries (a segment spanning a decile boundary is split proportionally). Traces are averaged within question × condition, then across questions (equal weight per question). Conditions: Group 1 (all correct), Group 2 correct/incorrect (same questions), Group 3.

In [ ]:
bins = np.linspace(0, 1, 11); lo, hi = bins[:-1], bins[1:]
ov = np.clip(np.minimum(segE.rel_end.values[:, None], hi[None, :]) - np.maximum(segE.rel_start.values[:, None], lo[None, :]), 0, None) / 0.1
ti = pd.Series(np.arange(len(tr)), index=tr.index).loc[segE.trace_id].values
ri = segE.role.map({r: i for i, r in enumerate(ROLES)}).values.astype(int)
T = np.zeros((len(tr), len(ROLES), 10)); np.add.at(T, (ti, ri), ov)
Tdf = pd.DataFrame(T.reshape(len(tr), -1), index=tr.index)
Tc = Tdf.groupby([tr.cond, tr.qkey]).mean().groupby(level=0).mean()
conds = [c for c in COND_COLORS if c in Tc.index]
fig, ax = plt.subplots(1, len(conds), figsize=(5.2 * len(conds), 4.2), sharey=True, squeeze=False)
for a_, c in zip(ax[0], conds):
    arr = Tc.loc[c].values.reshape(len(ROLES), 10); bottom = np.zeros(10)
    for k_, r in enumerate(ROLES):
        a_.bar(np.arange(10) * 10 + 5, arr[k_] * 100, width=9.5, bottom=bottom, color=ROLE_COLORS[r], label=r); bottom += arr[k_] * 100
    a_.set(title=f"{c}\n(n questions = {tr[tr.cond == c].qkey.nunique()})", xlabel="relative position in trace (%)"); 
ax[0][0].set_ylabel("share of words in decile (%)"); ax[0][-1].legend(loc="center left", bbox_to_anchor=(1, .5))
plt.tight_layout(); save_fig("fig07_role_composition_by_position"); plt.show()

### 11c. Role sequences, number of role changes, and transitions
* **Role changes ≠ segments**: `n_role_changes` = number of boundaries between merged phase runs. Its relation to segment count and length is summarised first.
* **Transition probabilities** are computed per question × condition (rows = current phase, columns = next *different* phase), then averaged over questions. They are shown for the four conditions; because only different-phase transitions between merged runs exist, the diagonal is zero by construction.
* **Order and position** are shown by tables (typical sequences, transition rates with mean relative position) and by the position distributions of doubt/verification/correction phases (sec. 11d).

In [ ]:
print("Role changes vs segments / length (Spearman within question x compute cells):")
display(corr_tab[corr_tab.y.isin(["n_role_changes"])].round(2))
print(f"median share of adjacent segment pairs that are role changes: {tr.chg_per_seg.median():.2f} (by condition below)")
display(tr[tr.cond != "other"].groupby("cond").agg(median_segments=("n_nonempty", "median"), median_phase_runs=("n_runs", "median"),
                                                     median_role_changes=("n_role_changes", "median"), median_changes_per_100_words=("chg_per_100w", "median")).round(2))

tr["seq8"] = [" ".join(AB[r] for r in runs_by[t].role.tolist()[:8]) for t in tr.index]
ts = tr[tr.cond != "other"].groupby(["cond", "seq8"]).agg(n=("seq8", "size"), n_questions=("qkey", "nunique")).reset_index()
ts["share_of_traces_%"] = (100 * ts.n / ts.cond.map(tr.cond.value_counts())).round(1)
top_seq = ts.sort_values(["cond", "n"], ascending=[True, False]).groupby("cond").head(6)
print("\nMost common merged-phase sequences (first 8 runs; S setup, R reasoning, D doubt, V verification, C correction):"); display(top_seq); save_tab(top_seq, "top_role_sequences", index=False)

trans = runs[runs.next_role.notna() & (runs.cond != "other")].copy()
trans["pair"] = trans.role + " > " + trans.next_role
trans["next_first_sim"] = runs.groupby("trace_id").first_sim.shift(-1).reindex(trans.index)
cnt = trans.groupby(["qkey", "cond", "role", "next_role"]).size().unstack("next_role").reindex(columns=ROLES).fillna(0)
P = cnt.div(cnt.sum(axis=1), axis=0).groupby(level=["cond", "role"]).mean()
cs_ = [c for c in COND_COLORS if c in P.index.get_level_values(0)]
fig, ax = plt.subplots(1, len(cs_), figsize=(5.2 * len(cs_), 4.4), squeeze=False)
for a_, c in zip(ax[0], cs_):
    sns.heatmap(P.loc[c].reindex(index=ROLES, columns=ROLES), annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, cbar=False, ax=a_); a_.set(title=c, xlabel="next phase", ylabel="current phase")
plt.suptitle("P(next phase | current phase), averaged over questions", y=1.02); plt.tight_layout(); save_fig("fig08_transition_probabilities"); plt.show()

c2n = ["G2 correct", "G2 incorrect"]
ntr_ = tr[tr.cond.isin(c2n)].groupby("cond").size()
tt = (trans[trans.cond.isin(c2n)].groupby(["pair", "cond"]).agg(n=("pair", "size"), mean_rel_pos=("rel_end", "mean"), median_abs_word=("end", "median"),
      median_words_before=("words", "median"), median_words_after=("next_words", "median")).reset_index())
tt["per_trace"] = tt.n / tt.cond.map(ntr_)
tw = tt.pivot(index="pair", columns="cond", values=["per_trace", "mean_rel_pos", "median_abs_word", "median_words_before", "median_words_after"])
tw = tw.loc[tw["per_trace"].sum(axis=1).sort_values(ascending=False).index].head(14)
print("\nGroup 2 transitions: rate per trace, mean relative position, median absolute word position, median words in the phase before/after:"); display(tw.round(2)); save_tab(tw, "group2_transition_table")

### 11d. Position and duration of doubt, verification and correction phases
Absolute word position and relative position of each phase run's start (ECDFs; curves further left = earlier). Runs of the same trace/question are not independent, so these are descriptive.

In [ ]:
kr = runs[runs.role.isin(KEY) & (runs.cond != "other")].copy()
fig, ax = plt.subplots(2, 3, figsize=(17, 7.5))
for j, r in enumerate(KEY):
    d_ = kr[kr.role == r]
    if len(d_) == 0: continue
    sns.ecdfplot(data=d_, x="rel_start", hue="cond", palette=COND_COLORS, ax=ax[0, j], legend=(j == 2)); ax[0, j].set(title=f"{r}: relative start position", xlabel="relative position (0-1)")
    sns.ecdfplot(data=d_, x="start", hue="cond", palette=COND_COLORS, ax=ax[1, j], legend=False); ax[1, j].set(title=f"{r}: absolute start (words)", xlabel="cumulative words", xscale="log")
plt.tight_layout(); save_fig("fig09_dvc_position_ecdf"); plt.show()
dur = kr.groupby(["role", "cond"]).agg(n_runs=("words", "size"), median_words=("words", "median"), median_segments=("n_seg", "median"),
                                       median_rel_start=("rel_start", "median"), median_abs_start=("start", "median")).round(2)
display(dur); save_tab(dur, "dvc_run_duration_position")

### 11e. What comes before and after doubt, verification and correction
For every doubt/verification/correction run:
* **Before**: the previous phase.
* **After / outcome** (descriptive, in priority order): *answer changes* (an observed answer change inside the run or the next run) → *answer stated (no change)* → trace ends → returns to reasoning → followed by another key phase → followed by setup. This is an annotation-based reading; it does not establish what happened cognitively.

For comparison, the final table shows how often a doubt/verification/correction phase is simply *present* in the window preceding an answer change versus in a typical segment window (base rate).

In [ ]:
chg_pos = {k: v.values for k, v in ch.groupby("trace_id").seg_pos}
ans_pos = {k: v.values for k, v in A[~A.is_change].groupby("trace_id").seg_pos}
out = []
for tid, s0, s1, nxt, e1 in zip(kr.trace_id, kr.seg_first, kr.seg_last, kr.next_role, kr.next_seg_last):
    hi_ = e1 if pd.notna(e1) else s1; c_ = chg_pos.get(tid, np.array([])); a_ = ans_pos.get(tid, np.array([]))
    if ((c_ >= s0) & (c_ <= hi_)).any(): out.append("answer changes")
    elif ((a_ >= s0) & (a_ <= hi_)).any(): out.append("answer stated (no change)")
    elif pd.isna(nxt): out.append("trace ends")
    elif nxt == "reasoning": out.append("returns to reasoning")
    elif nxt in KEY: out.append(f"followed by {nxt}")
    else: out.append("followed by setup")
kr["outcome"] = out
oc = pd.crosstab([kr.role, kr.cond], kr.outcome); ocp = oc.div(oc.sum(axis=1), axis=0).round(2); ocp["n_runs"] = oc.sum(axis=1)
print("Outcome of each phase run (row proportions):"); display(ocp); save_tab(ocp, "dvc_outcomes")
pv = pd.crosstab([kr.role, kr.cond], kr.prev_role.fillna("(trace start)")); pvp = pv.div(pv.sum(axis=1), axis=0).round(2)
print("Phase immediately BEFORE each run:"); display(pvp)
base = pd.concat([ch.groupby("cond")[[f"win_{r}" for r in KEY]].mean().assign(kind="window before observed answer change (incl. changing segment)"),
                  segE[segE.cond != "other"].groupby("cond")[[f"win_{r}" for r in KEY]].mean().assign(kind="all segments (base rate)")]).reset_index().round(2)
print("Share of 3-segment windows containing each key phase:"); display(base.set_index(["kind", "cond"]))
R["dvc_outcomes"] = ocp

# transitions into new / revisited / repeated content (lexical bands only)
trans["band"] = pd.cut(trans.next_first_sim, [-.01, .3, .6, 1.01], labels=["new-looking (<0.3)", "partial overlap (0.3-0.6)", "repeats earlier text (>=0.6)"])
tb = pd.crosstab(trans.next_role, trans.band, normalize="index").round(2)
print("First segment of the NEXT phase after a transition - lexical overlap with earlier text (by next phase):"); display(tb)

### 11f. Recurring phase patterns and correctness
Role **trigrams** over merged-phase sequences (e.g. `R > D > V`). For the most prevalent trigrams in Group 2, the question-level difference in the share of traces containing the trigram (correct − incorrect) is computed (a) over all matched cells and (b) over length-matched pairs only, because longer traces trivially contain more patterns. Repeated samples of one question are averaged before testing.

In [ ]:
if len(trm):
    tri = {t: set(zip(*(runs_by[t].role.tolist()[i:] for i in range(3)))) for t in tr.index}
    cntr = pd.Series([x for t in trm.index for x in tri[t]]).value_counts()
    cand = [k for k, v in cntr.items() if v / len(trm) >= .10][:12]
    for k in cand: tr["tri_" + "_".join(AB[r] for r in k)] = [float(k in tri[t]) for t in tr.index]
    tcols = ["tri_" + "_".join(AB[r] for r in k) for k in cand]
    trm2 = tr.loc[trm.index]
    s_tri = paired_summary(paired_wide(trm2, tcols), tcols, "a", "b").rename(columns={"median_a": "prevalence_correct", "median_b": "prevalence_incorrect", "frac_a_greater": "frac_questions_correct_higher"})
    s_tri_lm = paired_summary(wide_from_pairs(lm.tc, lm.ti, tcols), tcols, "a", "b")[["n", "median_diff", "ci_lo", "ci_hi", "frac_a_greater"]].add_suffix("_lengthmatched")
    tri_tab = s_tri.join(s_tri_lm); tri_tab.index = [i[4:].replace("_", " > ") for i in tri_tab.index]
    print("Group 2 role trigrams (S setup, R reasoning, D doubt, V verification, C correction); median question-level differences correct - incorrect:")
    display(tri_tab[["n", "prevalence_correct", "prevalence_incorrect", "median_diff", "ci_lo", "ci_hi", "wilcoxon_p", "median_diff_lengthmatched", "ci_lo_lengthmatched", "ci_hi_lengthmatched", "n_lengthmatched"]].round(3))
    save_tab(tri_tab, "group2_role_trigrams")
    R["tri"] = tri_tab
# transitions and length
rows = []
for c in ["nruns_doubt", "nruns_verification", "nruns_correction", "n_role_changes"]:
    v = cell_spearman(tr, "total_words", c); rows.append(dict(feature=c, n_cells=len(v), median_rho_with_words=np.nanmedian(v) if len(v) else np.nan))
display(pd.DataFrame(rows).set_index("feature").round(2))

## 12. Meaningful structural divergence

Divergence is detected for (a) the shortest vs longest correct trace of each Group-1 question and (b) the correct vs incorrect length-matched pair of each Group-2 question (`find_divergence`, three detectors defined in sec. 5c). Detected positions are **algorithmic candidates**; the context tables print the segment text around them so that they can be read. Whether the first structural difference matters for the outcome is not tested and cannot be inferred from observational data.

In [ ]:
dv = []
def run_div(kind, qk, a, b):
    res = find_divergence(int(a), int(b)); row = dict(kind=kind, qkey=qk, a=int(a), b=int(b), earliest=res["earliest"], rel=res["rel"], a_word=res["a_word"], b_word=res["b_word"])
    for name in ["D1_alignment", "D2_first_answer_differs", "D3_key_phase_asymmetry"]: row[name + "_rel"] = res["detectors"].get(name, (0, 0, 0, np.nan))[3]
    fa, fb = tr.at[a, "first_ans_words_before"], tr.at[b, "first_ans_words_before"]
    row["before_first_answer_in_both"] = (res["a_word"] < fa) & (res["b_word"] < fb) if pd.notna(res["rel"]) and pd.notna(fa) and pd.notna(fb) else np.nan
    dv.append(row)
for _, r in pairs1.iterrows(): run_div("G1 short vs long (both correct)", r.qkey, r.short, r.long)
for _, r in pairs2.iterrows(): run_div("G2 correct vs incorrect", r.qkey, r.tc, r.ti)
dv = pd.DataFrame(dv)
if len(dv):
    save_tab(dv, "divergence_candidates", index=False)
    summ = dv.groupby("kind").agg(pairs=("rel", "size"), with_candidate=("rel", lambda x: x.notna().mean()), median_rel_position=("rel", "median"),
                                  D1=("D1_alignment_rel", lambda x: x.notna().mean()), D2=("D2_first_answer_differs_rel", lambda x: x.notna().mean()),
                                  D3=("D3_key_phase_asymmetry_rel", lambda x: x.notna().mean()),
                                  before_first_answer=("before_first_answer_in_both", lambda x: x.dropna().astype(float).mean())).round(2)
    display(summ); R["div"] = summ
    fig, ax = plt.subplots(1, 2, figsize=(15, 4.6))
    sns.stripplot(data=dv.dropna(subset=["rel"]), x="rel", y="kind", hue="earliest", ax=ax[0], size=6, jitter=.2); ax[0].set(xlabel="relative position of earliest candidate (0-1)", title="Earliest algorithmic divergence candidate", ylabel="")
    dl = dv.melt(id_vars="kind", value_vars=["D1_alignment_rel", "D2_first_answer_differs_rel", "D3_key_phase_asymmetry_rel"], var_name="detector", value_name="rel_pos").dropna()
    dl["detector"] = dl.detector.str.replace("_rel", "")
    sns.boxplot(data=dl, x="rel_pos", y="detector", hue="kind", ax=ax[1]); ax[1].set(xlabel="relative position (0-1)", title="Where each detector fires", ylabel="")
    plt.tight_layout(); save_fig("fig10_divergence_positions"); plt.show()
    print("Context around the earliest candidate for the first Group-2 example pairs (A = correct, B = incorrect). '<<' marks the segment containing the candidate.")
    print("STATUS: algorithmically detected - substantive divergence must be confirmed by reading the text below.")
    for ex in [e for e in EX if e["group"] == "G2"][:2]:
        ta, tb = ex["tids"]; res = find_divergence(ta, tb); print(f"\n{ex['title']}\ndetector: {res['earliest']} -> {res['detectors'].get(res['earliest'], ('', '', ''))[2] if res['earliest'] else None}")
        display(show_divergence(ta, tb, res))

## 13. Answer development and answer stability

Observed from `answer_status` / `answer_value` (see sec. 5 for definitions), with the caveats of the annotation scheme. `final` marks a status, **not** correctness; agreement of the last annotated value with `gold` is checked separately as a *consistency check on the annotations* (normalisation is light, so formatting differences can cause false disagreements).

### 13a. Annotation reliability and coverage

In [ ]:
sv = segE
cov = pd.DataFrame([
    ("eligible traces", len(tr)),
    ("traces with >=1 answer-bearing segment (candidate/final with a value)", int(tr.has_ans.sum())),
    ("traces with a 'final' segment that has a value", int(tr.has_final.sum())),
    ("traces with candidate-only answers (no final)", int((tr.has_ans & ~tr.has_final).sum())),
    ("traces whose 'final' segments carry >1 distinct value", int((tr.n_final_values > 1).sum())),
    ("segments with status candidate/final but null answer_value", int((sv.answer_status.isin(["candidate", "final"]) & sv.answer_value.isna()).sum())),
    ("segments with status 'none' but a non-null answer_value", int(((sv.answer_status == "none") & sv.answer_value.notna()).sum())),
    ("observed answer changes", len(ch)),
    ("  of which numeric/other replacement", int((ch.chg_type == "replacement").sum())),
    ("  of which refinement_or_format (one value contains the other)", int((ch.chg_type == "refinement_or_format").sum())),
], columns=["item", "count"]); cov["% of traces / segments"] = np.where(cov["count"].index < 5, (100 * cov["count"] / len(tr)).round(1), np.nan)
display(cov); save_tab(cov, "answer_annotation_coverage", index=False)
ag = tr[tr.agree_gold.notna()]
print(f"Last annotated answer agrees with gold in {ag.agree_gold.mean():.1%} of {len(ag)} traces; agreement with is_correct: {(ag.agree_gold == ag.correct).mean():.1%}")
display(pd.crosstab(ag.correct.map({1.0: "is_correct=1", 0.0: "is_correct=0"}), ag.agree_gold.map({1.0: "annotated answer == gold", 0.0: "annotated answer != gold"})))
R["ans_cov"] = dict(has_ans=tr.has_ans.mean(), has_final=tr.has_final.mean(), agree=(ag.agree_gold == ag.correct).mean(), n_changes=len(ch), frac_refine=(ch.chg_type == "refinement_or_format").mean() if len(ch) else np.nan)

### 13b. Emergence, stability and revisions (correct vs incorrect, same questions)

In [ ]:
ans_feats = ["first_ans_rel", "first_ans_words_before", "first_final_rel", "post_first_final_words", "post_stab_share", "post_stab_words", "n_ans_changes", "n_replacements", "last_change_rel"]
if len(trm):
    wa_ = paired_wide(trm.assign(has_ans=trm.has_ans.astype(float), has_final=trm.has_final.astype(float)), ans_feats + ["has_ans", "has_final"])
    sa = paired_summary(wa_, ans_feats + ["has_ans", "has_final"], "a", "b").rename(columns={"median_a": "median_correct", "median_b": "median_incorrect", "frac_a_greater": "frac_correct_greater"})
    print(f"Group 2 matched cells, question-level paired differences (correct - incorrect); coverage-limited (questions need both sides to have the feature):"); display(sa.round(3)); save_tab(sa, "group2_answer_development_paired")
    R["g2_ans"] = sa
d2 = tr[tr.cond.isin(["G2 correct", "G2 incorrect", "G3 incorrect", "G1 correct"])]
desc = d2.groupby("cond").agg(traces=("has_ans", "size"), share_with_answer=("has_ans", "mean"), median_first_answer_rel=("first_ans_rel", "median"),
                              median_words_before_first_answer=("first_ans_words_before", "median"), share_with_any_change=("n_ans_changes", lambda x: (x.dropna() > 0).mean()),
                              mean_changes=("n_ans_changes", "mean"), median_post_stable_share=("post_stab_share", "median"),
                              share_gold_candidate_later_lost=("lost_gold", "mean")).round(3)
print("\nTrace-level descriptives by condition:"); display(desc); save_tab(desc, "answer_development_by_condition"); R["ans_desc"] = desc
print("\nObserved answer changes by condition and direction relative to gold (annotation-based):")
display(pd.crosstab(ch.cond, ch.direction)); display(pd.crosstab(ch.cond, ch.chg_type))
pre = ch[ch.cond != "other"].groupby("cond")[[f"win_{r}" for r in KEY]].mean().round(2)
pre["any_key_phase_in_window"] = ch[ch.cond != "other"].assign(a=lambda x: x.win_doubt | x.win_verification | x.win_correction).groupby("cond").a.mean().round(2)
print("Share of answer changes with doubt/verification/correction in the 3-segment window ending at the change (compare to base rate in sec. 11e):"); display(pre)

fig, ax = plt.subplots(1, 3, figsize=(19, 4.8)); dd = d2[d2.has_ans]
sns.ecdfplot(data=dd, x="first_ans_rel", hue="cond", palette=COND_COLORS, ax=ax[0]); ax[0].set(title="Where the first answer appears (relative position)", xlabel="relative position")
sns.ecdfplot(data=dd[dd.has_final], x="first_final_rel", hue="cond", palette=COND_COLORS, ax=ax[1], legend=False); ax[1].set(title="Where the first 'final' appears (relative)", xlabel="relative position")
pd.crosstab(dd.cond, dd.n_ans_changes.clip(upper=3).map({0: "0", 1: "1", 2: "2", 3: ">=3"}), normalize="index")[["0", "1", "2", ">=3"]].mul(100).plot(kind="bar", stacked=True, ax=ax[2], cmap="viridis")
ax[2].set(title="Observed answer changes per trace (%)", xlabel="", ylabel="% traces"); ax[2].tick_params(axis="x", rotation=15); ax[2].legend(title="changes")
plt.tight_layout(); save_fig("fig11_answer_emergence_and_changes"); plt.show()

### 13c. Group 1: what does additional reasoning do to the answer?
For each Group-1 pair (both correct) the long trace is compared with the short trace: more observed answer changes, later first answer, or extra words mostly *after* the answer had stabilised. "No observed change" means no change **in the annotations**; it is not evidence that the additional reasoning was useless.

In [ ]:
if len(pairs1):
    p1 = pairs1.copy()
    p1["extra_after_stab_share"] = (p1.post_stab_long - p1.post_stab_short) / (p1.words_long - p1.words_short)
    p1["what_extra_reasoning_does"] = np.select(
        [p1.changes_short.isna() | p1.changes_long.isna(), p1.changes_long > p1.changes_short, p1.extra_after_stab_share > .5],
        ["no answer annotation in one trace", "long trace has more observed answer changes", "extra words mostly AFTER the answer stabilised (no extra change)"],
        "extra words mostly BEFORE the answer stabilised (no extra change)")
    vc = p1.what_extra_reasoning_does.value_counts(); display(vc.to_frame("pairs")); R["g1_kind"] = vc
    print(f"same annotated final answer in short and long trace: {p1.same_final_answer.mean():.0%} of pairs; first answer later in long trace (relative): "
          f"{(p1.first_ans_rel_long > p1.first_ans_rel_short).mean():.0%}; median post-stabilisation words short/long: {p1.post_stab_short.median():.0f}/{p1.post_stab_long.median():.0f}")
    save_tab(p1, "group1_additional_computation_answer", index=False)

## 14. Repetition, restatement and new content

Lexical overlap bands are **descriptive categories of text similarity, not discourse labels**: *near-duplicate* (cosine ≥ 0.8 with an earlier non-adjacent segment), *high overlap* (0.5–0.8), *low* (< 0.5). Overlap alone cannot distinguish repetition, restatement, verification of an earlier result or correction, so the figures are read together with the segment role, the transition context and example texts.

In [ ]:
rows = []
for c in ["rep4_trace", "mean_overlap", "share_words_overlap"]:
    v = cell_spearman(tr, "total_words", c); rows.append(dict(trace_measure=c, n_cells=len(v), median_rho_with_length=np.nanmedian(v), share_positive=(v > 0).mean()))
rep_corr = pd.DataFrame(rows).set_index("trace_measure"); print("Longer traces within the same question x compute cell: more repetition? (Spearman)"); display(rep_corr.round(2)); R["rep_corr"] = rep_corr
segE["band"] = pd.cut(segE.sim_nonadj, [-.01, .5, .8, 1.01], labels=["low (<0.5)", "high (0.5-0.8)", "near-duplicate (>=0.8)"])
role_tab = segE.groupby("role").agg(segments=("words", "size"), median_words=("words", "median"), mean_sim_nonadj=("sim_nonadj", "mean"), mean_novel_frac=("novel_frac", "mean"), mean_rep4=("rep4_frac", "mean")).round(3)
role_tab = role_tab.join(pd.crosstab(segE.role, segE.band, normalize="index").round(3)); display(role_tab); save_tab(role_tab, "overlap_by_role"); R["role_overlap"] = role_tab

segE["dec"] = np.minimum(((segE.rel_start + segE.rel_end) / 2 * 10).astype(int), 9)
pos = segE[segE.cond != "other"].groupby(["cond", "dec"])[["sim_nonadj", "novel_frac"]].mean().reset_index()
fig, ax = plt.subplots(1, 3, figsize=(19, 4.6))
sns.lineplot(data=pos, x="dec", y="sim_nonadj", hue="cond", palette=COND_COLORS, marker="o", ax=ax[0]); ax[0].set(title="Overlap with earlier non-adjacent text by position", xlabel="relative-position decile", ylabel="mean max cosine")
sns.lineplot(data=pos, x="dec", y="novel_frac", hue="cond", palette=COND_COLORS, marker="o", ax=ax[1], legend=False); ax[1].set(title="Share of new content words by position\n(decays naturally as vocabulary saturates)", xlabel="relative-position decile")
sns.barplot(data=segE.dropna(subset=["sim_nonadj"]), x="role", y="sim_nonadj", order=ROLES, palette=ROLE_COLORS, errorbar=("ci", 95), ax=ax[2]); ax[2].set(title="Overlap by discourse role (pooled segments)", ylabel="mean max cosine")
plt.tight_layout(); save_fig("fig12_overlap_position_role"); plt.show()

In [ ]:
# Group 1: how much of the LONG trace has no counterpart in the SHORT trace (cross-trace similarity, same question)?
if len(pairs1):
    rows = []
    for _, r in pairs1.iterrows():
        sl, ss = seg_g[r.long], seg_g[r.short]
        C = (X[sl.xi.values] @ X[ss.xi.values].T).toarray(); cl = np.where(sl.words.values >= MIN_SIM_WORDS, C.max(axis=1), np.nan)
        cs_ = np.where(ss.words.values >= MIN_SIM_WORDS, C.max(axis=0), np.nan)
        for thr in (0.3, 0.5):
            un = cl < thr; us = cs_ < thr
            wl = sl.words.values[~np.isnan(cl)].sum(); ws_ = ss.words.values[~np.isnan(cs_)].sum()
            row = dict(qkey=r.qkey, thr=thr, long_words_without_counterpart=sl.words.values[un].sum(), share_long_without_counterpart=sl.words.values[un].sum() / wl if wl else np.nan,
                       share_short_without_counterpart=ss.words.values[us].sum() / ws_ if ws_ else np.nan, extra_words=r.words_long - r.words_short,
                       unmatched_over_extra=sl.words.values[un].sum() / (r.words_long - r.words_short),
                       rel_pos_unmatched=np.average(((sl.rel_start + sl.rel_end) / 2).values[un], weights=sl.words.values[un]) if sl.words.values[un].sum() else np.nan,
                       share_unmatched_after_first_answer=(sl.words.values[un & (sl.seg_pos.values >= tr.at[r.long, "first_ans_seg"])].sum() / sl.words.values[un].sum()) if sl.words.values[un].sum() and pd.notna(tr.at[r.long, "first_ans_seg"]) else np.nan)
            for ro in ROLES: row[f"unmatched_{ro}"] = sl.words.values[un & (sl.role.values == ro)].sum()
            rows.append(row)
    xs = pd.DataFrame(rows); save_tab(xs, "group1_cross_trace_novelty", index=False)
    sm_ = xs.groupby("thr")[["share_long_without_counterpart", "share_short_without_counterpart", "unmatched_over_extra", "rel_pos_unmatched", "share_unmatched_after_first_answer"]].median().round(3)
    print("Group 1 (median over questions): content of the long trace with no similar segment in the short trace (cosine below threshold)"); display(sm_); R["g1_cross"] = sm_
    ur = xs[xs.thr == 0.3][[f"unmatched_{r}" for r in ROLES]].sum(); print("Roles of long-trace words without counterpart (thr 0.3, summed over questions, %):", (100 * ur / ur.sum()).round(1).rename(lambda s: s[10:]).to_dict())
    gm = tr.loc[pairs1.long, ["rep4_trace", "mean_overlap", "share_words_overlap"]].reset_index(drop=True) - tr.loc[pairs1.short, ["rep4_trace", "mean_overlap", "share_words_overlap"]].reset_index(drop=True)
    print("Long - short within-trace repetition (median over questions):", gm.median().round(3).to_dict(), "| share of questions where long is higher:", (gm > 0).mean().round(2).to_dict())
    R["g1_rep_diff"] = gm.median()

def overlap_examples(tid, k=6):
    s = seg_g[tid]; Xt = X[s.xi.values]; S = (Xt @ Xt.T).toarray(); rows = []
    for _, r in s[(s.words >= 8) & s.sim_nonadj.notna()].sort_values("sim_nonadj", ascending=False).head(k).iterrows():
        i = int(np.where(s.seg_pos.values == r.seg_pos)[0][0]); j = int(np.argmax(np.where(np.arange(len(s)) <= i - 2, S[i], -1)))
        rows.append(dict(seg_pos=r.seg_pos, role=r.role, temporal=r.temporal_role, sim=round(r.sim_nonadj, 2), earlier_seg=s.seg_pos.iloc[j], earlier_role=s.role.iloc[j],
                         text=r.text.strip().replace("\n", " ")[:130], earlier_text=s.text.iloc[j].strip().replace("\n", " ")[:130]))
    return pd.DataFrame(rows)
ex_t = next((e["tids"][-1] for e in EX if e["group"] == "G1"), None)
if ex_t is not None:
    print(f"\nMost overlapping segments in the long Group-1 example trace ({tlabel(ex_t)}): read to judge repetition vs restatement vs verification"); display(overlap_examples(ex_t))

In [ ]:
# Overlap in the context of phases and answer events
fa_seg = segE.trace_id.map(tr.first_ans_seg)
ctx = {"all segments": np.ones(len(segE), bool), "doubt": (segE.role == "doubt").values, "verification": (segE.role == "verification").values,
       "correction": (segE.role == "correction").values, "first answer segment": (segE.seg_pos == fa_seg).values,
       "answer-change segment": segE.set_index(["trace_id", "seg_pos"]).index.isin(list(zip(ch.trace_id, ch.seg_pos))),
       "after first 'final'": segE.after_first_final.values}
ct = pd.DataFrame({k: dict(segments=int(m.sum()), mean_sim_nonadj=segE.sim_nonadj[m].mean(), mean_novel_frac=segE.novel_frac[m].mean(),
                           share_high_overlap=(segE.sim_nonadj[m] >= .5).sum() / max(segE.sim_nonadj[m].notna().sum(), 1)) for k, m in ctx.items()}).T.round(3)
print("Lexical overlap by context:"); display(ct); save_tab(ct, "overlap_by_context"); R["ctx"] = ct

## 15. Temporal-role analysis (`new` vs `continuation`)

Analysed separately from `discourse_role`. The annotation is **not** assumed to mean semantic novelty: the tables below test it against segment size, phase boundaries, lexical overlap, answer events and correctness, and show example texts.

In [ ]:
print("temporal_role values (eligible segments):", segE.temporal_role.value_counts(dropna=False).to_dict())
tt_role = pd.crosstab(segE.role, segE.temporal_role, normalize="index").round(3); display(tt_role)
segE["run_start"] = segE.set_index(["trace_id", "seg_pos"]).index.isin(list(zip(runs.trace_id, runs.seg_first)))
segE["is_new"] = (segE.temporal_role == "new").where(segE.temporal_role.notna())
nr = segE.groupby(["role", "run_start"]).is_new.mean().unstack().round(3); nr.columns = ["P(new) inside run", "P(new) at phase start"]
print("Does a phase change coincide with a 'new' segment?"); display(nr)
tb_ = segE.groupby("temporal_role").agg(segments=("words", "size"), median_words=("words", "median"), mean_words=("words", "mean"), mean_sim_nonadj=("sim_nonadj", "mean"),
                                       mean_novel_frac=("novel_frac", "mean"), share_high_overlap=("sim_nonadj", lambda x: (x >= .5).sum() / max(x.notna().sum(), 1))).round(3)
display(tb_)
ev = pd.DataFrame({"all segments": segE.temporal_role.value_counts(normalize=True), "first answer segment": segE[segE.seg_pos == fa_seg].temporal_role.value_counts(normalize=True),
                   "answer-change segments": segE[ctx["answer-change segment"]].temporal_role.value_counts(normalize=True)}).round(3); print("Temporal role at answer events:"); display(ev)
tr["share_new_words"] = segE.assign(nw=segE.words * (segE.temporal_role == "new")).groupby("trace_id").nw.sum() / tr.total_words
tr["share_new_segs"] = segE.groupby("trace_id").is_new.mean()
if len(trm):
    tw_ = paired_summary(paired_wide(tr.loc[trm.index], ["share_new_words", "share_new_segs"]), ["share_new_words", "share_new_segs"], "a", "b").rename(columns={"median_a": "median_correct", "median_b": "median_incorrect"})
    print("Group 2 matched cells, share of 'new' (correct - incorrect):"); display(tw_.round(3)); R["tmp_g2"] = tw_
if len(pairs1):
    print("Group 1: share of 'new' words, short vs long (median):", round(tr.loc[pairs1.short, "share_new_words"].median(), 3), round(tr.loc[pairs1.long, "share_new_words"].median(), 3))
dpos = segE[segE.cond != "other"].groupby(["cond", "dec"]).is_new.mean().reset_index()
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
sns.lineplot(data=dpos, x="dec", y="is_new", hue="cond", palette=COND_COLORS, marker="o", ax=ax[0]); ax[0].set(title="Share of segments labelled 'new' by relative position", xlabel="decile", ylabel="P(new)")
sns.boxplot(data=segE[segE.nonempty], x="role", y="words", hue="temporal_role", order=ROLES, ax=ax[1], showfliers=False); ax[1].set(title="Segment size by role and temporal_role", yscale="log")
plt.tight_layout(); save_fig("fig13_temporal_role"); plt.show()
print("Does 'new' mean lexically new? Examples (read the text):")
ex1 = segE[(segE.temporal_role == "new") & (segE.sim_nonadj >= .8) & (segE.words >= 10)].head(4); ex2 = segE[(segE.temporal_role == "continuation") & (segE.novel_frac >= .8) & (segE.words >= 10)].head(4)
for lab, e in [("labelled NEW but near-duplicate of earlier text", ex1), ("labelled CONTINUATION but mostly new content words", ex2)]:
    print(lab, f"(first {len(e)} found)"); display(e.assign(text=e.text.str.strip().str.replace("\n", " ").str[:150])[["trace_id", "seg_pos", "role", "temporal_role", "sim_nonadj", "novel_frac", "text"]].round(2))

## 16. Cross-group comparison and stratification

Group profiles (traces of the questions assigned to each condition) are descriptive; groups differ in composition (benchmark, difficulty, compute), so Group-2 within-question comparisons remain the cleanest correct-vs-incorrect evidence. Accuracy rates include all traces with known correctness (including truncated/unlabeled ones).

In [ ]:
cs = tr[tr.cond != "other"].groupby("cond").agg(questions=("qkey", "nunique"), traces=("cond", "size"), median_words=("total_words", "median"), median_segments=("n_nonempty", "median"),
                                                 median_seg_words=("median_seg_words", "median"), median_phase_runs=("n_runs", "median"), role_changes_per_100w=("chg_per_100w", "median"),
                                                 first_answer_rel=("first_ans_rel", "median"), share_with_answer=("has_ans", "mean"), mean_ans_changes=("n_ans_changes", "mean"),
                                                 post_stable_share=("post_stab_share", "median"), rep4=("rep4_trace", "median"), share_new_words=("share_new_words", "median"),
                                                 empty_output_rate=("empty_out", "mean"), marker_missed_rate=("marker_hit", lambda x: (x != 1).mean()))
cs = cs.join(tr[tr.cond != "other"].groupby("cond")[ROLE_P].mean().rename(columns=lambda c: "mean_" + c)).round(3)
display(cs.T); save_tab(cs, "cross_group_profile"); R["cs"] = cs

# G2 correct-vs-incorrect stratified by benchmark and by compute level
sub = ["total_words", "p_doubt", "p_verification", "p_correction", "first_ans_rel", "n_ans_changes"]
if len(trm):
    out = {}
    for b, g_ in trm.groupby("benchmark"):
        w_ = paired_wide(g_, sub)
        if len(w_) >= 3: out[b] = paired_summary(w_, sub, "a", "b")[["n", "median_diff", "ci_lo", "ci_hi"]]
    if out: print("Group 2 paired differences (correct - incorrect) by benchmark (questions with >=1 matched cell):"); display(pd.concat(out, names=["benchmark"]).round(3))
    cm = trm.groupby(["qkey", "compute", "correct"])[sub].mean().unstack("correct"); cm.columns = [f"{f}__{'a' if c == 1 else 'b'}" for f, c in cm.columns]; cm = cm.dropna()
    out = {c: paired_summary(g_, sub, "a", "b")[["n", "median_diff", "ci_lo", "ci_hi"]] for c, g_ in cm.groupby(level="compute") if len(g_) >= 3}
    if out: print("Group 2 paired differences by compute level (question x compute cells):"); display(pd.concat(out, names=["compute"]).round(3))

corder = sorted(df_t.compute.unique(), key=lambda s: (0, float(s)) if re.fullmatch(r"-?\d+(\.\d+)?", s) else (1, s))
acc = df_t[df_t.correct.notna()].groupby(["benchmark", "compute"]).correct.mean().unstack().reindex(columns=corder).round(3)
print("Accuracy by benchmark x compute level (all traces with known correctness):"); display(acc)
display(df_t[df_t.correct.notna()].groupby(["difficulty_tier", "compute"]).correct.mean().unstack().reindex(columns=corder).round(3))
fig, ax = plt.subplots(1, 3, figsize=(19, 4.6))
cs[[f"mean_p_{r}" for r in ROLES]].rename(columns=lambda c: c[7:]).plot(kind="bar", stacked=True, ax=ax[0], color=[ROLE_COLORS[r] for r in ROLES]); ax[0].set(title="Mean role share of the trace (%)", ylabel="%", xlabel=""); ax[0].tick_params(axis="x", rotation=15); ax[0].legend(fontsize=8)
sns.pointplot(data=df_t[df_t.correct.notna()], x="compute", y="correct", hue="benchmark", order=corder, ax=ax[1], errorbar=None); ax[1].set(title="Accuracy by compute level", ylabel="accuracy")
sns.boxplot(data=tr, x="compute", y="total_words", hue="correct", order=[c for c in corder if c in set(tr.compute)], ax=ax[2], showfliers=False); ax[2].set(title="Words by compute level and correctness (eligible traces)")
plt.tight_layout(); save_fig("fig14_cross_group"); plt.show()
save_tab(tr.drop(columns=["raw_model_output", "thinking_text", "rtl_labels_json"]), "trace_level_features")
save_tab(segE.drop(columns=["clean"]), "segment_level_features", index=False)

## 17. Findings, limitations and further questions

The "Observed" block below is generated from the numbers computed above (nothing is pre-written); interpretation follows. Everything is observational and exploratory.

In [ ]:
def pc(x): return "n/a" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{x:.0%}"
def f1(x): return "n/a" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{x:,.1f}"
L = [f"DATA: {R['n_eligible']:,} of {R['n_total']:,} traces eligible (labeled, complete, correctness known). Min samples/question used: {R['min_n_used']}. Groups: G1 {R.get('G1_n', 0)}, G2 {R.get('G2_n', 0)} ({R.get('g2_matched_n', 0)} with matched compute), G3 {R.get('G3_n', 0)} questions."]
c = R["corr"]
L.append(f"LENGTH vs STRUCTURE (within question x compute): median rho(words, segments) = {c.loc[('total_words', 'n_nonempty'), 'median_rho']:.2f}, rho(words, mean segment size) = {c.loc[('total_words', 'mean_seg_words'), 'median_rho']:.2f}.")
if "g1_role" in R:
    g = R["g1_role"]
    L.append(f"G1 (all correct, long vs short, n={R['G1_n']}): the extra words are distributed (median share of extra words) as " + ", ".join(f"{r} {g.loc[r, 'median_share_of_extra']:.0%}" for r in ROLES) +
             f". Median {pc(R['g1_frac_count'])} of the log length ratio comes from more segments (rest from larger segments); long trace has both more and larger segments in {pc(R['g1_both'])} of pairs.")
    L.append(f"G1 answer position: median first-answer position {pc(R['g1_first_rel'][0])} (short) vs {pc(R['g1_first_rel'][1])} (long) of the trace; median share of extra words before the first answer: {pc(R['g1_pre_share'])}. Pair classification: " + "; ".join(f"{k}: {v}" for k, v in R.get("g1_kind", pd.Series(dtype=int)).items()) + ".")
    if "g1_cross" in R: L.append("G1 novelty (thr 0.3): median share of long-trace text with no similar segment in the short trace = " + pc(R["g1_cross"].loc[0.3, "share_long_without_counterpart"]) + f"; median within-trace 4-gram repetition difference long-short = {R['g1_rep_diff']['rep4_trace']:.3f}.")
if "g2_lm" in R:
    a_, l_ = R["g2_all"], R["g2_lm"]
    L.append(f"G2 (matched compute, n={int(a_.loc['total_words', 'n'])}): median length ratio incorrect/correct words = {a_.loc['total_words', 'median_incorrect'] / a_.loc['total_words', 'median_correct']:.2f} (medians). Role-share differences correct-incorrect (pp, all matched | length-matched n={R['g2_lm_n']}): " +
             ", ".join(f"{r} {a_.loc[f'p_{r}', 'median_diff']:+.1f} | {l_.loc[f'p_{r}', 'median_diff']:+.1f}" for r in ROLES) + ". CIs/p-values are in the saved tables; treat as exploratory.")
    sa = R["g2_ans"]; L.append(f"G2 answers: median first-answer position correct vs incorrect {pc(sa.loc['first_ans_rel', 'median_correct'])} vs {pc(sa.loc['first_ans_rel', 'median_incorrect'])}; mean observed changes diff {sa.loc['n_ans_changes', 'median_diff']:+.2f} (median paired diff, correct-incorrect).")
if "div" in R:
    L.append("DIVERGENCE (algorithmic candidates): " + "; ".join(f"{k}: median relative position {v.median_rel_position:.0%} (candidate found in {pc(v.with_candidate)}, before first answer in both traces: {pc(v.before_first_answer)})" for k, v in R["div"].iterrows()) + ".")
if R.get("G3_n", 0):
    L.append(f"G3 (all wrong, n={R['G3_n']}): median pairwise role-sequence similarity {R['g3_sim']:.2f}; median within-question length ratio {R['g3_rr']:.2f}; in {pc(R['g3_conv'])} of questions at least half of attempts give the same wrong answer. Answer behaviour (% traces): " + "; ".join(f"{k}: {v:.0f}%" for k, v in R["g3_pattern"].items()) + ".")
oc = R["dvc_outcomes"]
for role in KEY:
    if role in oc.index.get_level_values(0):
        L.append(f"{role.upper()} outcomes (share of runs, by condition): " + " | ".join(f"{cnd}: " + ", ".join(f"{k} {v:.0%}" for k, v in oc.loc[(role, cnd)].drop("n_runs").items() if v >= .1) for cnd in oc.loc[role].index))
L.append(f"ANNOTATION: answer-bearing segments in {pc(R['ans_cov']['has_ans'])} of eligible traces, 'final' in {pc(R['ans_cov']['has_final'])}; annotated last answer agrees with is_correct in {pc(R['ans_cov']['agree'])}; {pc(R['ans_cov']['frac_refine'])} of observed answer changes look like refinement/format changes.")
print("OBSERVED (generated from this run)\n" + "\n".join("* " + l for l in L))
open(f"{OUT_DIR}/observed_findings.txt", "w", encoding="utf-8").write("\n".join(L))

### How to interpret (to be read together with the numbers above)

* **Why correct traces differ in length** – the G1 decomposition separates *more segments* from *larger segments*, and the role/position tables show which phases hold the extra words and whether they fall before or after the first answer. Whether the extra text is new content, restatement or verification must be judged from the overlap tables **and the printed segment texts**; lexical overlap is a proxy only. Longer is neither better nor worse here: all G1 traces are correct.
* **Correct vs incorrect** – the paired tables compare structure at fixed compute and, separately, at (near-)matched length. Differences that vanish in the length-matched analysis are length effects; those that persist are candidate *structural* associations, not causes.
* **Doubt / verification / correction** – outcomes are annotation-based descriptions (answer changed / restated / followed by …). Check the base-rate table before reading a key phase before an answer change as informative.
* **Divergence** – detector positions are candidates; the first structural difference is not claimed to cause the outcome.
* **Group 3** – similar labels or lengths do not imply a shared failure mechanism; the answer-pattern table only shows what the annotations reveal.

### Limitations
* Annotations are used as given; their reliability was **not** audited against humans. Answer extraction can miss candidates or flag formatting variants as changes; gold matching is a light string/number normalisation.
* Only labeled, complete traces are analysed. If labeling failures or truncation correlate with length, difficulty or correctness, results describe a selected subset (see sec. 4 and 6). Truncated traces are not part of the group analyses.
* Group membership depends on samples per question, thresholds and the compute levels sampled; questions with few samples can change group by chance (an all-correct question with 4 samples may simply be one with a high success rate).
* Repeated samples/phases within a question are dependent; question-level aggregation and bootstrap CIs are used but small group sizes give wide intervals; no multiplicity correction. Wilcoxon p-values are descriptive.
* Lexical similarity (TF-IDF cosine, 4-gram reuse) cannot separate repetition from restatement or verification; novelty fraction decays with position by construction.
* Observational data: no causal claims about phases, extra computation, revisions or correctness.

### Further questions
* Are the structural associations in length-matched Group-2 pairs stable across benchmarks and compute levels with more samples per question?
* Do revisions *toward* the gold answer differ in their local context (doubt/verification wording) from revisions *away from* it?
* Does manual reading of divergence candidates confirm the three detectors, and which detector best predicts a substantive divergence?
* Would a semantic (embedding) similarity change the repetition conclusions, and does it agree with human judgement of "new content"?